# wheat-benchmark-d2-tiles-validation

固定validationの画像でEfficientDet-D2の4タイル候補を採取する。枠の評価・採用判定はローカルの固定評価で実施する。比較元は提出58370638。

In [ ]:
import hashlib
import json
import subprocess
import sys
images = {
    '010dbcc8f': 'f2ae6bc819fe3373bd3d1da8a81a08e55eead65d2d66d086833b0a66e6f1600c',
    '0114c88aa': '84709eef92543e2c4c3dd6ee9047aac47ed3f79428203ce4dad3ab3bf1aa996d',
    '013669953': 'd303590063b6bc3bda874fade14b42e0bcb22dfbbda4bc2d1b515cb65ad41363',
    '01397a84c': '94078de14023cfcaa29fae9c98e04cc8f43cbad4df8ef9602661ef0b0969b64b',
    '02992044c': '52be3c1d7b44c4685a5e182a29f6b1b1391abfeeca29e0cea45f89a88538778b',
    '030314634': '935a4c86f4f57397d64821240bafbb8d6e636f6d46f0ee02c1c0d61c9c1129fc',
    '031ed396f': 'cef7bc783617671293e947bea46d0d4982b6ba1d790cf6e02ed700510464bbec',
    '03448724b': 'dbfc35d45ae586ee5c0207e149b0c14d5608e161b5834fa6f1bfa679ca40ea29',
    '037d96952': '09081c1a19a22bb2afc679d4d1707e0c36dfb0806369415f805d67379e4573ce',
    '03a177626': 'c0edbb1c5ff1a650c418d6302a9c05efc61c81180a85b324fc279faed18d1d0e',
    '03a988adf': '66c3318eb84ef10778ff9e7576d4549b520cecf55448c42408739c307e64afe1',
    '04f906b5d': '7c2b1e2e52421dd8cc14aa4d4c617aff79c1f5ad5f08b99e6df9b7236e222b20',
    '0573ed37c': '3e3c0bf2ada0167ae25c46da75a33a9e6f9439d99376b4479463d2eb3bd990ad',
    '059125072': '033f42061152bd7be5515288ea1e3069884251d572cf23f505bc88221fea9304',
    '06152e5bf': 'b2198d9c040b0bf4e556f9eb3e63e53976f6bfe7597304e3f5784666fc9f5e11',
    '0787635a2': 'e5e5106e73bd80805335268a538cf59b6b969fe159bd5c14384d23a7e0c0bcbc',
    '07a4d2b9e': '187c6e7120660af06317605f11af11af32d83433a68ad7165b13dd7370ed3e01',
    '07ebc6045': '9f3cc5201fbdc249d5931f64d4e3e528b40ff4326f840dff26e08b5d700e374e',
    '085d0d511': '0bb732979591e13c220e705f380bd45d39cbcc956a6ae4e2e91aa235da670911',
    '08a65db34': 'e119b93d5dece3ab41cf004845c544276dd63b605eadb59a7179459f4e7a51cb',
    '08aa26ac1': '1ea6aa6a288c98b57913cce98e0f3c4993a2617ad25ef0e51534c44286189e6c',
    '09f97e0cf': '467917b70858b232a2a7fb3af6b26bac113ccc96989dd0e200a69cf46f67f7c4',
    '0a10cd864': '25ea3301580520a72d1f395bb290720bf7c966d9dce26a82d7aba4c4ecb81628',
    '0a73f8a0f': '291c307d8d07a4e5db3f353efae233a187a68b795bb0fe32870736e5af7b4f2d',
    '0c264c98e': 'ff95782fdec08752fb5587d2c42053372a71e8f4f4d20bd555d518aae521a14f',
    '0c2d20111': 'f51decea34deeaf504e345ae99a5f6195c82c6fbbd20a8ebf82176303c000cc4',
    '0ca410614': '235fc17eed9a80c7f9996ba4328b8178baab652d1e135d9f816ea7532b05f06e',
    '0dc2de69d': '485cc2daf1733a516a4bef64df9a6ae3171060b956e35e4a95d104b5bb82453c',
    '0e0809a3d': '6c810b6d36cf658b654b6e68c7dc672ae954adc73093ed0afba69856a636413e',
    '0e86615d0': '2cc7ede6760f9583d3affbd98b6f792c1a4b08247f2c4f9c7398f0adf120f6d2',
    '0eab1baab': 'bd7d9be3eec7ccfd31a4c84417cb57d1c1e9387860feed152b82e51072ac52d7',
    '0f0bb7843': 'b4b6bf2f45be6df4062faa24465982c6299eb8b43e2b2ad6a7dce20ba64dc2e2',
    '0f4911203': 'bba0ce948128a0f59d030e2241810266402e0c990b49c4e95123e66fbb2587c6',
    '0fb8ec104': 'd11176a205c2601d43210bef9f8ae3677e3f6786558cf2ddb1677897566b7125',
    '0fd4ca0d4': 'b3929da5479dd79da93e5cd654625e5d53f2df2eb2f0355f6469f37ae1897c16',
    '0fd63ed98': '3104234d22ed3af398763bb683c70fdcf830abb5ec7fe99cfaec8189c2568f0d',
    '0ff60b2bc': '7b1201660f2e68ac43a83b778c2f863a255dee2d9439e80dcc9ff2bdc49b24c9',
    '11182d7ae': 'd071caf2aa9d68cd0ee252d2118e989bc5a040108df3a20c723282c9a1f0793f',
    '11c916a8a': '88a0f4950553406f16f96b10f772b69a2b5d739e21d00d1efaf298feb26e48fd',
    '124820ed4': 'dd661c44a501f20b14c8f4e804b3490b0fbcfee8812cf6dad81d5e09a12a9092',
    '12a15fe77': 'ddba57f5f149dc8f1bd94ae06ba62c951dfb493fc628d8ecad93d805f7aa5bab',
    '12d014f3e': 'f181b06b318506202ce2ccb5e73a1b77e4fda029b8b5df99794145d07acd2588',
    '13395ae85': '8e093fefbac77c1fb4397e4865610f4d944073a8b49025ece51f8a13edd1cd9c',
    '13d56dca0': '91e624568c35cc7915f60f6241f958d29dd61e8f0ce6caa2e2321471fd3a3641',
    '13f7f0e24': 'a436cd23e68469f7bcbe7dbc50637a97cc8d976e04d4940a3e2f07f90960bc27',
    '1443b3f7a': '4238b710f96e5187395303b0c2b73c1ab55a7ca04b1257d45c7a9be40386ad58',
    '14d024cc4': 'dfa3781ed8d0ef3748690b885595a58d2c8f18906bbe6da7ba789cf36259a074',
    '15153bb54': 'ea40b6233527227983a914b631434232f5f814a75eda3820b6b22c49342d754a',
    '15486e5af': 'b6508493a92a19fbcb44b1bbcde8e7cf7a5a44d3d11dae969b277b79d745c5a1',
    '15babe6dd': '69c48c3aa52328be31bac5f7835a01d9499d66d5d1684130b0418398729a08b9',
    '160cdf8e1': '253bb86cc31cfe7580cb651995d31fea2affdbdacdf807ad73ae276493f445b9',
    '161f20a53': '85159919acfedfbb544fd2cdaa4a25a047fd5bcab953a2ad7f505ea96eb360d8',
    '16a0dc111': '892f5a59a891bd361a47e645219932e79966be1abac14e33ee124945be142d2d',
    '16e075fbb': '151027b3eb3029e1b22792f28e19b8bbe38ed893a706f33cc5c7ac2103497ddc',
    '1734d7a9f': '82349c35175610c629b0a3ec99f7df845f3c5d24a016bbb55f652890f7eb21f0',
    '1794c0a24': 'eee206b3fb02c6a8d089934d92d1f9d01e56c8133b088c3caf24bbcf21d6ee11',
    '18df0fea9': 'eac0a36af79c1967d952e1e5ba4351ec998ef196fb259e9ebffac2de9d4c206c',
    '19255ce1d': 'a4dcb45c27499d48f4446dbefc43bfb1416c9602683a235d5c8c639476de85b2',
    '19e5ab184': '7b9d7bb61fbf3c4908bb07a1f9a1f7eb5b61f7341007ab5a9980ff9310958d2a',
    '1a94773a1': 'a9b3870cc2045700e35f6dac6b2cc1e345f61b719c028ef69dfa7517bc847c38',
    '1b2a30a59': 'd64b0fd6dd5951991145b7b551d788fd71f703c7d1d1e52a42be2ff6eabfea7c',
    '1b3376c60': '9edafa345acd8e3940c5a1fc77029051975a325a1fa3857aa8cd8521f93c82db',
    '1b547b682': '728579d1bba2bf8012408d8f505b598afc09504b397f23fda775e244724a5962',
    '1b5e821f9': '803090e223deaee5e18565811075a420c7b0c535bdb39c745f51c9fb70aeaff6',
    '1b99d2fd6': 'd7b4d9b6d5508448079b1c22023fe75d9ae1cd658bb40b370f673c9bf13ff5cf',
    '1bcb0e91e': '032fb3033ad9c83a6ec9d85bd872e8158ed8959998dbd368eef0ccb8605eb896',
    '1c2c0786b': 'ad0e872dc762c2c6a76d471d86265821d329af96b9808fe376e27010bf2bfacd',
    '1d554c619': '84e312cfd41de0fcb5d6eea389cd63923e90da8622dadfb795d417ac3789f6ac',
    '1d74edc1e': 'd8444b9320d13236643520b3c0acd7b9c8c8848c8b7c4fa54c656ac8944efcee',
    '1eb2bad31': '262d008a3a7ff493c801478e82a8086b0c7d16635f1319539c95013e4a41c7a1',
    '1ef16dab1': 'f71f0db316c4a9515d78f3dc390a6c25f6e5d291dc65595fab3caae2e7478da6',
    '1f548e2f2': '3dc6662beb2ac0ce8dd1c6724ee2cc317be6aee0241c486e0e3c507f7379602e',
    '1fcc8f8d9': '7a64c2a582006ff43ac04bca28b3e567e0dd6f364fb810c5867861820e303bc5',
    '20697ca8f': '5c9f5abdec210b3227632262f45c52bfc750235d17f6aa8ae09d7fcf7bfc34c4',
    '20e9f1548': '1d311f495f3c7e55385cbc54e7d8e89ed875055e151c4552395c4f65e897757f',
    '217c8fd61': 'cd60e75a1b64091d622d631a76495dd96c18537399b2f6e5de374a4be41fe015',
    '2204976da': 'fb3ae009cd6a5962ffad2ede0c74fcbdc1477c4bbbaf997fb1d2d8a24aea12ed',
    '224017b57': 'a83c6e4673fc585f6f86b46bb7dc02c5d6a6a83b9806617ba85cec63f0d7ffef',
    '2277ae74f': 'bbe4d7c0cf3ec45647ef66bbb6fd9c7b2c55ccdfcfc8de5f474549a31624d5a3',
    '227971e53': '4b47cb3711d07dc4446459d995c76e38404d7688129757784c0a59c3d61336c5',
    '229716799': '607281a872efd00aba69b64da79c337ee27a0b3af7f78c43633f53108bfee6ae',
    '22dccd0b7': '344952969f76a4a62bb506322d7d7e01c9e7a4a6650d86d5bc79b0bae712fc11',
    '238c6cea4': 'b4640b14def88559c264667efb238f8ad5d9a9ed7ac81741ed0d3d87148164cb',
    '23e3b5130': '5347fab97fae298e147acf4db05b4443e4bcf7bc770d2a2d65d0fcdf3e628008',
    '24a90b0dd': '4565ee0669a70f015acb3533e346a92f8144b4083a9ab42e07b4bd355e117da5',
    '258c0d943': '848cd9c6a7182500adec1c96d74e2d5b16df8dd5b7fb9e0a212eae77f0992ee0',
    '2661979ef': '25d37f39a904025932d15e61497d3fac1e42e2cf34e97b81609f5906766ee65b',
    '269a5f98e': '35e7efd78cdf0da8f7f33d2efb253c48977ac3a9f326c3acab028aa3b0e9c16b',
    '26b7c3188': 'e3253a82bafc91f5f01d430632b5a99042d127323553009482012004a6ea7f2e',
    '26eada42f': 'ad037805418953ce7970cf67ea6dfe9132cd27eb1cb41716fcae02928efa0f07',
    '277fa7280': '82f4f65e9e38c4fb7125644456d64095f7fd49c24f1c7602c6225c6faaaaa788',
    '277fff409': 'e6e0d0fcc2ea76c8edca06e11af3d156430d534d8b36ee21af22649e5f8bee31',
    '278038027': 'aa1a00da783a5c23bd3fa5d27cc0f6c6e4778d673ad0558b5eecdd64ec525c85',
    '28ee29473': 'e93b923cf9f2b977eca4b93d37523f7d8f851d6ffcf0edcc826eccc0bf378654',
    '291fb66d1': '08881fe120e7c24b76e4928a2571d8cc95a85e1b556ed9930d4367bdbcd3f1d6',
    '293a1fd62': '6d5b756bcf0e6987d8a96cc811eed68ef28eb8e5b368caf9e4e89feda0754dec',
    '2aa35dade': '526955367e07d7af486fe6dc46ed62ed913dc91363ebb7fa78b2cd46ac7de37f',
    '2b32a4cc4': '3104b204473acca0160c338670e8e167e641a17841c5b8c23f22027926f995ac',
    '2b3c77755': '42a0fadf96fc0108df1f479dd4f73303f677d274d6fc85ede84b35fd90084898',
    '2c7638ee8': '0a398f4896543e11e6a728a7e36822822098e1685b8c7368ffac652320c38ae5',
    '2c836cccb': 'ec16b06a58cf72d813489012ba3e53b0a76739a50d78be7a6dd4f0859df55d0f',
    '2ccb37ea9': '292ec8ace16d1532f55d6f748676d4519e5bfa430ceda95670040facc68bbee5',
    '2d0fc19b8': '8765b45676eecc40e266643d8368ced9368161d3646038e2b4847a45fb0ba7cb',
    '2d1b6ce2a': '081b80b625fcd8b8041cc671cc154836b559ab2855c49e85a29f5b47fb3c6963',
    '2d3e3f783': '69e12e5a9bb3d0aae72db0b70eb6e5eb0ba239310298608a2645db8671ac43bb',
    '2eaac7a41': 'f36c7304358b1550f6fb469e89c8951dac8ac4b4e7b9c384169f279f2db38ada',
    '2ed43720f': '54a30620947653ef9e85528373dac1e4e1542e82a8bfe216eaadf8dd95cce701',
    '2f9755f66': '0ca0fe258c92899ccf2d102aec10fb1308123869fa0b777c9a4d285a4ad4680d',
    '303def58d': 'd0a649b49b7f2e9e84ca7834839a52250a34fc3eb937348079087e5092f36fb9',
    '3049b3745': 'e6694dae781b2a58afca88b9daa2a2472d94d51fe232f64b06b458b5506c94f4',
    '305bf8b23': 'b7166b10c222474bd941321673cdbf5211d7a11791795fc471c89fb00fcfc83b',
    '30e5213ff': 'd8050ed68044c5c9d7e92e23bc7efaa7b305cdb1321410ca188c9792ba884db7',
    '311498261': '2576007b975bf74f560486b056c9f977320ca2f7d873b27588285bd82a15046c',
    '31c085c60': '2a621f5342a92dc5029adbd89b843d06590c15c2b54cffa892b818354e0c35ed',
    '31eb0625d': '164b87f643ee97af669af78bff7595e6894a30543f07b19d43969e501112d3aa',
    '32494cc53': '2619055ecdfa9f25408bd980c83e42439dbc803b247f11d97999f8eea53a2b1c',
    '33c69a58a': '88d69aff33a165c74d4163af35d63285362dfde3ea7031c4f889ea1a1caced52',
    '340ac8d52': 'c475a6198b0284f2f6bea9cb0dc5000c1e15bda1f4918b76b7854cb8510531f6',
    '35b59e0a0': 'c52c43a60168e25cb7860fb00d81fbe55e5877d72d0ba6e95674f5f63b16b084',
    '35f84d33a': '3f5240d9c71a0afcfd8e46059f48541baccb9a3f07f440a8d0e32145255040a5',
    '366187e59': '2ca5589549770d769ed243b58c16986f652a4e98a3e7944846d13556779c177e',
    '3733d0e53': '44ab2a8b9a3c4e221606851a6038fa4091e902dd87b5400c4a6830bc464acee7',
    '3741873d5': '7aa897393daa8aba5be1770930f99ca4cd53afe484171d19fdcd16ed84d61630',
    '3744f2482': 'b4758e16f8157c1850dd23f403f76f055a9e31d5ad89380aa87655355035464f',
    '3753c58f7': 'd07cddc0f9707e7b9e3aad7b15d8d0f44516e84bce9257f11a3b8305e690d1c6',
    '375a4795b': 'bda996529e2440a52ee85d252b148679e38adfa6104367c697c0efb6dcd2e129',
    '37b0d5ea3': '3c9bbef0a01496001d7c9774fea1daa808d9e6f1305091009795d484146ea8b4',
    '37bcaaeb6': 'f9af3123d4a1596303190142e5b5d3feca424f85f7242ccceb5d53e853067644',
    '3834dbe25': '879ceffcd29e1a426a8c9b8ec380e006d70d669734cb6b873003f103632bd38f',
    '38b259a9e': '2bf54dff7faa9ab6276034652328a2ee64fe1d2e54f8c9f61794a8eae51ea33a',
    '38c8180c0': 'b1521854f09f13273ea0cc27cf53dde8fcacaa0ea50993b07b3cfd1ba77474c0',
    '38ed976da': 'a07ae8c6cfa75fd92f715506e91ea8f2ac3a420300565b48a1c518fb3991c6a0',
    '38fd197f0': 'f56c912549f6be60160e5bdd7e295d3920458e459332bb33365a78f128fe8e9d',
    '398708128': '29aa0ebb35fef52dd874ae2767c254121b972b7aec2d489e642cb1589bbe4dd9',
    '399fce7aa': '678a8e1c8f9776f6d96852e831037828db9bcb5343ed8749aaff6a37d148e9cd',
    '3a061fb14': '5c2a4df0106cb02f25912890ff9b8c9d9958bc07f5dd7a7d58434b6a8a987c92',
    '3ab9d0640': 'ca168c863bba8dfdc0901d40ca847888134cb728f9e5fc0c07ab0d03f8cab1d3',
    '3ae057d35': '8640fe07d1c0074eff155544d14bf0db7e0c753c8e41cbd75da49191ff8ae11f',
    '3b2183a3c': '075e2828bde38307d385315e529ea1ba18d64950d6e7ee36aa9b6801e13b8621',
    '3b4f7ba0a': '34f4cda70e7e165ae266a50432e5e21a9262695946a8fd30ec74174770401cb5',
    '3b778c35e': 'aff0675dc05ab5eaa4c2ac3971779b7028ac55c293703af2a531b3be0891c376',
    '3cf4abfb6': '1a6ac98d40522e1c579271f4b4d66fd4c34815d7d7fe6706fc6827327defdfcb',
    '3d74e4614': 'f6ca95d2e5185a3bd9157e45f5def99ea93ca92494029d69627e4c12037f7b05',
    '3d765f241': '62f7f7be885b1e016d1b64c017bc9e2151f084df9aad6bb6c8bb5976551ad7ee',
    '3d9125515': '910a5b6b3999680e9d094c7584ef7a2ca903d13df291bf3576be316824eb1695',
    '3dd0a7c0e': '53226eae23123d1c972c7c0c4bf769b759d96cf50b5892a030aecb93fe7e2e82',
    '3dd44c524': '863db5d15ad6bb1a82e79d304e79bc993912612302de5a5553a03395759a4a6e',
    '3e0213b6f': '1b019b7f81e3e3bf9ae6460123a32a4e5dd2e3b49c0fa4375c6ccfffc6d1c7ca',
    '3e5cf1ed6': '6f845fdb97b769717b0ec916b95963b1a52fc6690db36dfcab45a149f40c192a',
    '3e6110840': '4a243777d431183561110a5e09ae841819cfccdecdd8b2242edac0b60ffd534c',
    '3ead80a57': '62a9d0b02f8babc7dbeb5e18f2336fc0b9e239713a602290eb05a35328c99b46',
    '3ed7c19f0': '299165ac03e2db5552d380ad087f2de546a8ac4e9ff4fc89554f9864d648c9c0',
    '3eda8f481': '6cd1daf33134ab49152bf14ee07d4e9944c3804c52ee9c6cd067e7882ac89e79',
    '3ee6fd90a': 'b9a3d470e7eb18e84341fbc024f8e259eb6713c81bd10201d28f316fa9b3db72',
    '3eece2a34': '95239204a0e599f436aefce0c8fa17517080f8d06674ad55b68620d84bc26b4b',
    '3f2eddbe8': '8667773f314c43eebe63416ce8c962f1e29def65f23b25bfb9ca6f8f3cc1d97e',
    '3f6898489': '5b415781f17eaaad282d74e770acb9c24f31f2c5c043c7a1dc35304c25af3286',
    '3f8f6b1a1': '7c1bca9959ffc58ca9524b6eb86b37a52e6194f47352a760dd9d7ccb796923a7',
    '40cdb5389': 'd57996821e17d115bcd4acdaff9b18d5c1ee443221f7120342d53cf1955a8c70',
    '4112a34bd': '619505c12d5f2b9d55abd9fb6af0328906b8e27074814c6bbcb73e75054e6865',
    '411d24167': '0c72087e3d08d11cb56a4d19ef8d33a9df7690f25f9da4a52a96fad2171911aa',
    '4227725a7': '50af98179b7f68fdb41f20b09a347270758e643bd08e697888023cfddf717018',
    '42f5fe6a5': 'fd2a32ac9eee885aa0eb4e1ade6c40f696874ab86b18d7fc09a82218c0fd8b0c',
    '432d13b1c': '6ecdf5157e4755c46e2e3647d660060606c8d70313953dece6a2ec1bb42714bc',
    '43d29ec95': 'aaa7b821b6f3276b90cbd838baa8c66e82513c2d7cd2346c1c95d537dd04a664',
    '44afd181e': 'fa106201d86088876f868f633bc6199550da1104f3ad456e70fc66565018bf23',
    '44fb6b8b2': 'e8542dc29dd2b24a4e74708c48f12a9ff18cf77fe2227127cef9d0a291dd2fc1',
    '452859b3a': '6525843d551107967dbb2c6cafb026e20f6edf2d6ddbd8d357c38de744a6eecf',
    '4532a128a': 'df17771f789d39729cfda3a64ba0e15312db7be89cba795c90091d8905daaf57',
    '457752437': 'eb2a0f5fac8c94cd170e5af746e9f0517b2b47cde7ee99fd88683a2a56309ee6',
    '45958be5e': '25e3d255a949d3f4e4f13dfd8f480fe48f0b0f9c2e9f0c68b77f45984798829d',
    '463a95dac': '0e95539f7b456ec564d3e723316d9b4d251263909a6f377ef63a91207c12703f',
    '467e7283e': '7c992d3bdaeea190e80d0c31ee577f71b7977c020cd4dd0e18c191ca370d9afb',
    '4764548b9': '1daa1446793ffc365b384f4687a6d61300455266ad722f7b48ce7d7f23158de6',
    '478e21230': '5c866c0b6aa148c843f5e1c0c6c9944dc1006579159ac60e72e1f7aa10b68187',
    '47b0d504c': 'd948f89c2776e742523142dc941633d1c766c0393842492cfe63238b39a879f3',
    '47dcd5b34': '3e33140fa36cecf7dfc97444fbe8c28f052f1b22c8fad7c22ed38d44a0466cd1',
    '48bdfec34': 'e40267c2bb0d793b2de069079e29148ff70a33eb3a44db429e72036b6d0d2713',
    '4972b9493': '9bd2a8009e10b545e6be2286188fc366240239b222b3c744a02943fea8e7056c',
    '49d76c63d': 'fa6fc58f1440fc27062f5dff144424be97edd8477b8a101a132970cb48eeee62',
    '4a2662fae': 'f165b5262f6a46d239e347d80f249c96be567667893e05402fbc96facf33dd61',
    '4a29d08ab': 'e3a397e750670b8cafbe24b26c7df3c86b94051d3606688aed3b4383b77bfaaf',
    '4a304fb45': '4d38db9a2cb2f5ef411965af6b28199a6e843e353c54e6c9a56fa1c64b626114',
    '4a44720ca': '315bf52e377820e2622fa28cec9d02f8998019f6494dab49b0980bfe48377286',
    '4a654c127': '3e7b8f37092a0d5c7475fe829f3f704cadab8d9bc8945739e61a06cfea31d8f1',
    '4a7825bbe': '49c788284147730926f80d67f2eacc82f80afc2e84a8514fa4794df2c342e546',
    '4a875aa50': '3cb90634b5113ce2c1372a87a738bfc5083f1211002b23351109d6fef96bc508',
    '4b4f6de9b': 'c3473bc545518ee0131154dcc58b0aac76275e078e619168c22533febc223845',
    '4b74f7320': '5de075a8c79f21cbc51b96584e3322f9cd9a312a4a18ea015e294d3d7f97625b',
    '4be09242b': '73d2c95ce78833e088d6d34cae2880abde02eb7d7932ed3b7650be6e7e57711c',
    '4cdcd49d4': 'af9f7d3e5a54be1b77e368801d917e652b8af02709ce6fc6697038e6d74d2fc9',
    '4d103d0aa': '38a92e9dc74e13a8a182430756eae122b9a5479be14b450e1560967be703b191',
    '4d57ebd90': '73152a7f4edf1ad93c7b5e4f99f9e2324d8c951b754e3c9989a7b4edf51f2bb9',
    '4d6921225': 'fc783bfc82daa3b47a72a9656be6e18ef984aead2a8abf771f80abaf70ec1c74',
    '4d77c47b6': 'c4552b9dbead3737bbca4412437fd39672dc4c357e8246b834286912cf1fbf23',
    '4dd820b52': '488356c3f07310d050ea8907e3406ed4cc4b7717da3e9e21ad38566b812cf1c7',
    '4def855a8': '1674cb5b38660552889e117b697b098073d95dac4bcad94f9c212cd23cebce1a',
    '4e9096ca9': 'e98fdfbe70b48bc970bb9ca8de34a28b9c2052fb6219c9f9894c2b9d961a580b',
    '4eb442168': 'c610d6230a7109f81dfb09f6484b54caaa4e1066e642df988ea8d06b4518d678',
    '4f13bffbc': 'f9c11aa9d98bf10b84d454cc386120ae0fab2b74e067fb0422ea89d3109545a5',
    '4f75a8d52': 'ae067b29576e611d7477f813153fa178d1a8145d05b439638d7698a61b723860',
    '4fb926950': '0b8699ece707d652b65c11d9453e3428116e89889991714f552214f868c76550',
    '4ffeedce2': 'c41a114fcc96002e2ab5f030fc97bbf0887d2c75bdca8a8d08f2ed32d1eb9685',
    '5016adc4b': 'b4daf4f67bdcfbac748e1eedc6cb0cc6162b07ada4f622b39675de98118f2477',
    '5041b4b98': 'e3f55fc3e02eed146c82ab49ac94e2f1a9cef40263a2942ac26cd7030614c388',
    '50ee1a4fc': '9c683a0d4115c17cbd8ab0403dbc50803e4936e88243bd512771c0d105e0d439',
    '511091599': '55c5677041fc04f273661250204d514452fa12a9ad1b905330ea4905af7a8eb5',
    '516218913': 'f84bac36a88add5871d26969092389998725fad4aa73f52c7d8ce56dbffc7496',
    '518796b7c': '23fe9070e43b7879b139ef5fc46c26a185f4f2ff99f82444b5c68becee28e1c2',
    '5187b2d69': 'deccff6fc4f988111ddf6ba3cba3643b5c4e65f8eaa74924f81b4834a682340f',
    '51b4a91cc': '09940fea04df93778212c7425d475163d1489d35040281db51ea3985a9600158',
    '51cd1fe14': '110c9cc9ee97c6e5dfd715adf06a18aba2f7a7108e4d2fd2a45565c8266392d7',
    '51d7e0c3d': 'f5f9c1f19de9d19944f5a670eca7f7aa0ddcf8a549c28d25a5757cf1e4cef316',
    '51eff11e9': '32b7ad81757d284cfe89dbab77dd28816e8894b65a09d1eec33e86b60d6b6bc0',
    '522edf5c3': '4edf3f9e1e773140383ef851f84e25cd481f1bf6ca0445fb361fa7c44c0148c2',
    '528621931': 'e207153a8e98a17984e1dc82874caa3c7cefd092df2b0365a8f6cf06666967ae',
    '539ee1a56': 'c8d93ea207f6ef198cba7582aa2098c5758bc393d60cda5b01b3d6ce9c635b7f',
    '540b11d4a': '3985071b203dd2dbbef34ebac2061d6b031360202ac0cf587617b3ba8c5642d4',
    '5498d5aae': '308ca4c01d861bb7a9a88dd27b81079e91ac8d619b6071227ba45f1e4e478b60',
    '5559af3a9': '75037e15be234befb7619c38ff0e1f724ef844293134c0dc59196d82535eaf21',
    '555f4fc58': '2b05b7bfe3c8baf2416723e56af282a6e3b15630c4029b44a4bea522a6c24acd',
    '56105ed87': '76c42db3b57c3d4ae2f8f52c14aaf952b887f9740103e454f3a1ab9f75869fdf',
    '566c90045': 'c2127c81fc5befb1ea429aeb5edecf4879894072a38da0891d4725def3d02d3e',
    '56c1bc7f0': '94ea6d6ac2dc6a6d7d16fe516e1046c0f1cbaa67afaddf95f94e90ff087051ef',
    '56fd2d9c7': '971146cb03635faf9c1810849fe1b1bae08674771652feee61ff5f2668c8a98f',
    '57ae1baf6': 'aab541638e1f4ee479ad70101cdc54932077e2c52c20a76a6bf4918ce7395bc6',
    '58c6beb2d': '89b3ef67c707d7f4f71f12389599b0b3c7e66eb9e23aa654c223842fb911caa6',
    '59ab1cdd7': 'e24bea617a8ef35f19a622b3185517b92dff2bc76309919fe7b2159bb6df5122',
    '59ad70280': '7970cc927a66414c178821ddd6a97cda61d50a2eb9e5ccb14238b2a2f099df5a',
    '59ade64e1': '68936baf6ec73ed13dc7943bc74b39a296ea1f612c2c7f0a331139f028cffd15',
    '5a28cf3a1': '2279f8cd1130f9e83908b82c34072a14344b2a0fa8c3d8deeac5222d3435d17e',
    '5a4432266': 'ee0a49eaaff335afde7b8b64d3be70725113d9d2e0ec09004ed7316a3986a002',
    '5a5631a48': 'ef7039f05cc6f6989e2450b9472e1108012c440d8e4ed3eb022af5d4f0b4cae4',
    '5a77d4b5d': '12d716a2b93f6a22035b491c46a5c48d7b2f04842324527db40ed27b19068c36',
    '5baa23af0': '935297e13addc5209629470521865dc50092b4abc422d9b1d1d5930fc76ec498',
    '5c2e16f27': '80af848dea91070292ee86cdc6e9da32d074655124bf3f1f4aff4ce76f150a7d',
    '5c6197826': '5cc6ca6fbae24aab4117b4ab0810ff9181df29745b6061e4dd8bf51a415e224f',
    '5d31186c4': '70e7d672aab09475b91eb39a01bc8772f65136fcbc38f9377b0cc3dfcd08f1d6',
    '5e0747034': '795576f1699207d762bb83722560143f981acd9c070794b1a48223e8bd1ccdcc',
    '5e30f8f4d': '521c32d54b655e6e46fa25082a52d3d8d76073a8660797403c498ed88fc180d8',
    '5e9cd13e4': '8b7125e1dfbf29a3e4487ec4f527184d96411cfed6943846591799808b3f87b1',
    '5f9151e33': '0718a604f970d75e5b3644ecb261b59d170b7b3be5e519b78d5a4a2beddce668',
    '5fabe93b1': 'c8baeace25912da49921fbf15caf9299bea38a79951f0c2fe3456f6a5b273671',
    '60913f4db': 'c055038f251680dcb452500763ddb4bafc29fe0f4918b8e631a16ade1a9eaabc',
    '620d8294a': 'ad0433a76bfdc75b0fcb02c0f699db62ac45fbe1fd6cd925705e54dd73a6d784',
    '630b12b1a': 'b7638bbecaebe7cc32c47c098ba7d776250c14b74a7eba5e5ad9e22d6aef9473',
    '63467d323': '2dc1b7c65e83f49a4625efe9902e6324966358a2b753a4773c3e95f615204135',
    '6434a5bc8': 'd933bd6b43eb792ded5a92d4341d2023351011ac1320599833959fe43c83697b',
    '654a741f5': 'a19791c0841a80b3df825ee98db2a07cf9915b651052389120b85772717757dc',
    '655ef2d6b': '487a837bac3c5c9c46febb10519e7ab2f42f924d7a40fd82b4d3381033911b72',
    '657f6bf98': '8a8bbf457e04fd9ad69f5fa97de1b92ff11958981d683cc9825f04108f63e2ee',
    '667b4a999': '8ac3b20021526b81722e2992ff8a2ea10e86b9288328a86d44e7c8101f0a5b87',
    '66c96dce0': 'da5409ce67b763172c923563dc103504eb80d26e565933d0979daeb8d9f3c97f',
    '66d91891d': '168f800ec6a0d962fbc497a10bda25ec138c01a0edd4a2ccc06f329f8a66e359',
    '66ee3048c': '1726d7bb14e836b189507768b53d9e15cab40cc7218fb90e4738ed6f92cf818f',
    '687972820': '118805d5cbc8e63500226cb7a823492309097d0cd92e1652dd81b68def0a5723',
    '6895c563e': '591ea6aa631864ef15670b7b18c62519f9c8e42f2f486cb0642c0604e466cb9a',
    '69286a6cc': '73275c074e78e4e3eb2d80d15bf807a697fbb7ceb9fd667542b1f31d22435508',
    '695cd49dd': '18cb437e46c72e2bb18e0345ff8cbc90771594d53c0881d0c4a62b70dcf3c057',
    '696530284': 'fd7fe34f69f4556fc8aeb9c1d9e6b3b5376442396678b4cb7db84879b882a9e3',
    '6994e1821': 'd68fb8eb88c2fb4e1fc9453a66f9419d4af2f5ec6260e130c67aa7e4b62ce69b',
    '69a211ccf': 'b0a5f0b0d7657af2d89aa264c0144c4fe38ea40e8db4622e6a6388c5a8cb9820',
    '69e509038': '6b669b50fa3ab1a027fae8895b7440aace986ea010dcac366ea86d30e5201f50',
    '6a0e4e8cf': '957e3e51660cfda79a770af0e01494c37c91402c97d546856c5e19ee7ca8e698',
    '6a614e72e': '553376d67e7a542e7f30965ac05832a420c969bf113f8670a4c94c95aa901d39',
    '6a7cbd408': '09d0144f0bd1ce49a1fba53f221fd99ef6cae35168e0f4dd097849497124b2d1',
    '6a8a08409': '99535d7a105004ae70bc7acd069934153f93a889aeaad20055f2958961c47b4f',
    '6b4169a72': 'd798e4230426df6a300597977caed20411498fa2985441e053a287b06f05a7db',
    '6b62a043a': '5c9d0f62fd64d6506c086692ec31f363d232ad0b0256e307a1564de1d743257a',
    '6b8416819': 'bf30603b5e8e5394823324ad94a2f06f09960d7f792ac6cabdd645315fe6b058',
    '6bd3f949d': '7feb0247cae30083ffd54f04572f2495c522950e179431c546b378abaa74e515',
    '6c071b9da': 'f156442680682e7a7be5b53edd58dd2d5ab371dc42844e2cde0dbf1836a4bfe0',
    '6c31301ae': '119d32be7a416ad219fe0cc1f3e51ce5ea991ed0cbedd9f89504863b4a9b67ad',
    '6cecd54f5': 'ad13fd5f39fbd62f507b8f611b01bc1f33766a8ed08b2f6fbda31817c64bced2',
    '6d1ff0f34': '30f3720189b4b10013a9de2ed9e4a162e9b745ce97171d9de5bf9354c1a4fd57',
    '6d993d18d': '38a890809e2193e2ef9be8cb58ed555b30f27f7c43ac95c9592d428616299727',
    '6e43d71ec': 'd5c8829005da10a640ed6c4f0890df807268cb7be6a3f1612cdb9372d2afb0c2',
    '6e69bf773': 'ced8c15e93356ab7f57514ca64c5755d83e090992a57b6a171a1dd56ae25af26',
    '6e8842770': '0b02ed90a89f49b5f371e92fb8f42e8b54cf6086eb7b2047f1d67319a9beaa8f',
    '6ea32891e': '2fc0a5eef27d0c0072415497ced13bf02b417eb0df789cf218822fa0c0ea4c02',
    '6ef909f8f': '20a05bef85f82ff403096a9d015cd539724f89ca246697cf3e0541feccc6d6a5',
    '6efc51980': '8453fcf24b446adcbed46f5bc1b64b30c84194b46856f36026f72090dbf50bc8',
    '6f9bcd471': '3d591e81d4fc2980e490aeab4d0efb85d37b46ee7aca4195b56296b5bd101697',
    '70bc24fa2': 'f275c1b240c92857b60642e1a79d320d806bf52f36ea464e880e329e0fa387f0',
    '71d2a4b3c': '85dd282cab83e3ffce9d66a92b7ffc20bfd4c06eaea7b07bf25b508109c8506b',
    '7227652d3': '4432cb805494fec9faaea30d98e5ea2fb2f7a7995e9bf236ac1dd569eab40321',
    '7252dabad': '562ca124c4748deb717fc83756dbd47669774f51bf73c8b180cb2af6cde8aa12',
    '72c3714bc': 'd1a9e601ff88810474d2d9cd2736a58cc177ea63ba74740e01327e4089ea8404',
    '72fa543d6': '46eb9861b89c0bca7796a5a38a5b268220e0ce5453134ebf258fd51201e02ad2',
    '737f28df2': '0e839b0b4b0450d8c07ff989cc836ec73b25dc330f52162fe8e2902f17982562',
    '73ed111df': '8bf9824b3a6646c257447cf551acb7762c323a5d7363953e7ca1faf0c51b78f8',
    '740d3c904': '3a28a7f079e09e70ac8616ad233e5fe95bd0ef7b9e4db3e509849f7636d04a99',
    '765f1b36a': '9b4f4dc3057ee37ca0e44d7cbb05bb3cee60f844701e93472532c2855b1ee2c6',
    '76b0b1204': '1632aa8bdbd699f72516694e8aa156d5b40db721e0108ccf476f2cbafac2f5aa',
    '772287113': 'e3f0fad3e90262ff33ecb1ad01df3dd2462ed98562266c2be09e0222516f0f7c',
    '773c33268': '5df2b77c964a8f988de92dcf8feab0e90d7781d8683f6370c654407abbc2d6b6',
    '7820f5412': '2c7153df03734171c8c3770ed7a9d657573e6a1429eaa02ca6c8b2fee181301d',
    '787d3091c': '2fa36e7bf38e2a94006bfda14955048ea6c1e73f0bc3e73d66aacdb7fa78a618',
    '797dd3e01': '1889546cee4809f7819b8ae7c034f7cd0e8930b304c26f09ff10df390a3dc703',
    '79adcb57c': 'ca9c51e93ec7e212df4c6a392d8087e4ed9e697afb017019acfc06ca1bc69545',
    '79b825776': '281b7f20bb39113f9337498173bba9f7e8f140c7d62367a6f84c99af580f0783',
    '79ce67248': '27f173f772f23942659d18e666cfa8a05b85eac8155f56a437a08fcef3d76fc6',
    '7a9d1d1f2': 'c6b49889ab188184a53a6b6c9404ddd40bcdc458cb4c55232437a595e77d90c8',
    '7aa682e0f': 'c346bc68e1f0e2e188376e014cbf4401bd15b5a79ece35d7ce02ebaf57e0459e',
    '7ad46c7f4': '2913c58bb0311b2611eed1af360d99331e9bbb4867f33045cdadf9c832fd3999',
    '7bf6e7991': 'fbde241248f936b347cc9b5e922f44ea32ba405a7ab1417e54d0669ed42bd172',
    '7c23b7b20': 'd79324de5d21e5513a57c6b8cf995c2bf717ce96cfdaf9ff97c392658172c764',
    '7c598ce6b': 'ce0e6806c5878e2e41d24be06059812f9938ef3dd5f3cbdf9d35be06f4765732',
    '7c659d49a': '8fa3ef9d92e5be1f81177b7cf4a9bac16711e78974d0eb52bf7f3f130384fc62',
    '7cca42bf6': 'b2d572818de6deee1706ae4b618561a9dedab187a3196b0dd7ccbc50967930d8',
    '7d5af5b74': '4cad8da8061ec38c0bcd9a156858320671ad13bbdc5c273dad1331cd06df32af',
    '7dac36eaa': '87395bb9ff16131d3e51a1de2e4353dea97c0fe84eddf0773b6403c0f1d42f36',
    '7e18f16b8': '140cf5946ffe2a59e7cfeb7607268d6f8ca77b401bc902cf0e715b4148731ad1',
    '7e6a307fd': '9376ee6252b265933ecf7f3ffbc684771607f36fab2bfd0c7178722645cd4c73',
    '7e73872ad': '526dc65351d4a9615ac7946f716a697f3b09a2cd3fcfd50c973ec60a99a0ca01',
    '7ea16f78f': '937017b500b0409ccc0a5b1446b8d62aa0cbba6cf8b4cf77b3cbb16dea2acdd5',
    '7efb28abc': 'eacec356b9366cd252f9ad679fe14ea9978803744c22bd5f79dd35c0faceb55d',
    '7eff8d967': '14a27e9c7427588c799a75b6570f4f824ac5510f7b7d262ca1680994df97f5d8',
    '7f01525b1': '90b9dadc8355bcb0458f5cbf87f5e8b46a000720c1416e54013576f2a293e2f3',
    '7f0e525f3': '8d8d55dbd7863193b5685c071f0b89f25182a7eb7d686246822c56d350b24d9c',
    '7f4d07cc0': 'cab55e52268af816eb748ce1440301ec0c9d355526430fe95b7cab1164e8cc12',
    '810aba893': '83764fac754c36e115cbb1b3b8faececab41754d623dec0f0ad4ab1b7c9627d0',
    '8115795f7': '60bd8495b2fd0eb981e3584a4435087317ef43915a2cf4c79152e992077cbe39',
    '81b76f372': '65f3fe0483a4aab2fab684ca22c69cb5cf3223ce7d72d35758efe2f464faa8dc',
    '81efcfbdd': '00699e2e57b068cfaf41de8e615502340c892deb042189e145cd315564b18b08',
    '8200547cf': 'f28bb29c492dd9c606a834820e68854437d71a8f8190f060d036cbaa294dde8c',
    '833577238': '38c6d991d3c08930ecf13861d9460ef62d8d4bc98a5e3f399aa858cee46f8475',
    '848e992bd': 'cd0d205fb5f7ab5263b867eaf5eee6ce15a9431c6e22ca999ba880167476e367',
    '8498f5a6c': '5c4f0be739d92fed1ad422b17a928a800917b8768747b4b447a07f9838bbf06e',
    '84cdd23e4': '39fa6de1c868494ffc622933a803fa63c08923b64ae45a2c37737255e052060c',
    '84d992b88': '71dea9ec97255a326377d1d2ca091f6e6ba378ce52dd607fba96da1dc68e4f77',
    '85631af25': '79ba23c4201894ae140e6b6941da52ca13f9a362e7a8bd1eb66c7eef76c95140',
    '85e3ce681': '5c0436dccd54e19357e60cfca7db79ba208a17af13298e20f18e434c9e6b57ee',
    '86426e386': 'e7e2c50b613b84717746ee2c33678eb26f9c8408701aa7c1f7757a826d4776a4',
    '8764d55c1': '6e7aa5c4fca6b3c035816e0a0033387d74a4cb96bbc74b559509e64f3fc87796',
    '876d458f2': 'a5138e9e246ec235e1b37dd03f6418bd000f95ec7805d8ada1efd4791d40a594',
    '879d8a9d2': '35119dc4d06e65f6009207bfdcfd18d26de6b4121f7fe811dce4abc0bc367969',
    '87a0373b2': '8b78c20fed4ac88277929ef3a9dedd875e678e02caeb9a56b531e5778ecb6ecd',
    '893938464': 'dd9fe0b9e30a0b69ce39f6034a2b661dea39070febd492adb85219f30d57d7c6',
    '89fb3f19c': '36fe891e4465f3739ee801a95187b2ae56782a2999622d1dc18533384998fcc1',
    '89fe13786': '11586921fd49b9cea49a5bbce7aa038ee6a6fdb4f87af9c57f4c8431cfe8fd09',
    '8a24885c6': '89d651eff67fa4a48ae07f633efbdde9ff0922ee7937d2adfdc628ae3d24789f',
    '8a3543c3b': '58cebbba43ad509b41e9c293d1d48df1ce110b512af9c6616e0276a92fb50d40',
    '8b89a4833': '794792b2d6eb15e8f67d08f61f58e953d3b795ebc1e44cfb1948e7675347467d',
    '8c5a33b46': '0657b656dd2954f0764a46bb98d8a3518f4ecd9eac8fb38ffc5eec8ff4647dc9',
    '8d0b36c74': '329d5b0985d1b893175fd6eb6725850c00f0bce753d59911fe00b4295d54df4c',
    '8d4f8b1cc': '4166008da20f499a3e5b0323394b5e736f628aba5ba3a45ad6865f75934e626d',
    '8d9aaa54d': '250eb857a36386cb36c3aa1ed9a3be06a2a6e72683e122598aff5d9d78bea4ee',
    '8ea56fe8f': 'addcdf52f43e10b82f4bf0ad267a7c874e841ecc50db6d783cc91171c0cafb6d',
    '8f507b3f7': 'f2be410357921d0b9a3ca60e92ec6aef8d518677a1ae8a65ff0f5345881a7728',
    '8ffe2f23a': '7b50bd905a9655a89660ae2fb9abd99d80b6d6f4e49f0880377939fe582ac90d',
    '90184122e': '2dba6c394d332929581b1c44df71edc39a48decc3b02352e728d921481c507f3',
    '901e23e9b': 'bd3c6bf0e9f687d45b7352de7e55d204fea8c5fc8271497c2b2fbcde8c71ab98',
    '9052c4be4': '42e580396f7e30570d52a9b0cbcb8471859d3fe349a3515986b6b3e765e00b0b',
    '91c7fb84e': '367d56e71f5697cee34998b4cf68f9e194bc2c497fa39c7b83be4cd2ff14e57c',
    '91ef6ef2b': '200ff9126a530d014e7fb297adf60494460802b735332cefe14c7f26e29c3659',
    '920d0cdad': '1c4ccfa993947b0ed8173c8c6828a3e881b4869ae506e61e33eed033be88b776',
    '92c083d34': '1bd4f4b0251e6fb38c5a63bfe839b47ca5a39799f54ee47d17931c8b97206f46',
    '92dbf3418': 'afaa5f8f22ddd8e69777d97d056d3d4893f85022e7f53bcc64c077167f58e7e8',
    '93bad1af2': '6f157b367c34e75050436c3709ddb03d666b0e6b2f7552f098a0e5e98bce0857',
    '93c9d721e': '99ca494bb00f4e79510a290e6710668a34e3b4e1b09f1517cd0550b03da93442',
    '93d056da6': '6304c2dc521cc97aa486c22a064b3d761583b40181d11a236f3d270a5b08d9e4',
    '93de65f55': 'ebc96532e6b6b11edb29f8a764957fea3d55d9acc11d893a35eec5dd1af45c2b',
    '94265d02a': '0e7923f0ddeffb2213f60fe29c5f29d390b9590b23afaaafc733af280b3f2145',
    '947117f47': 'e4b1644caaa8b3254e3d0a6e914cc2c1f001a773f0a8a640d174e37c992113d6',
    '948c989ef': 'e9418e6cb9f25664145498b933cc3f06f9ece8fbf9a6bb3f72bdbb98321d6643',
    '94aee84d6': '2a3800b23849985aebbb968c2e62598bee8c426e2235bc6e67d6413f2c009004',
    '94e3ee08b': 'd1ace7d7b1e2e3b1f180c7e89ad7b30c730991d1d58d1808fec99ed19ab8ac5e',
    '95ee952ba': 'd1b37bbc2e613ff8c86dec1c7e1449eaf1ccb2837dd458cc303aecdd021dc1ae',
    '9613afcb6': 'abe37cc3622d71e9f924e7e0c84ff837ba803e3a2c36dc8141e5bf8a8ed5f37b',
    '96367fda5': 'bf0216539da90914cb16424501c5e721332fd0a989c3ca6d020ac43064cafc42',
    '9658b9c3c': '790ebf8ddd214e86f4d00763eaa684db82e35e014377f32c9e66f6500cca0ae0',
    '96b19d664': '2a5506d8e92c518006942b11e70f2cad39e43972bd9e1cc9fc1e59ec45fee161',
    '96bf13ac5': '7d0e38e92aab3b361f4485feb1ca95b513e54dbc5b52cff0827bd1d1d203c931',
    '97788b3ae': '7db3f0dd1529a8a21e0f60a8338c68954431eb371ba95682effaa2dadb823876',
    '980c88da5': '3b4fe861226072590c9b6152d0643d174ba6881a32209ee37fdacfbebb0141e5',
    '9832f05e1': '35581ba7e8766c640c90de95a08f0ed03e833194cc0736282d12dbf8815c257e',
    '988a60e6e': '6038850c9eb35ff8a42bb037516b71387a390c4000d1bcc6d004a44eafc0545b',
    '98ce36a93': 'def9dc56cfd2c99d5745df4445ff46d0120d5104b5ae81305cf48c4efd5efbb8',
    '990c1777d': '08bc3950f674a9642297ddf187206f2beceb568d49a7a3f04e38bcd928068e17',
    '99886a700': '978d7f4ab4ecbdd9c5b8ff1bd7fe2becb801b152edc45f5cbd3758f05e280aa9',
    '9a9f42302': '7b06391bf6c0b7cd5b92aafe0577584a81da509b32af1f4dbc53a72673a2f3d8',
    '9aa5dec56': '4df7b4f64f869c57f1406491b3f0303dd20ece1845db3e652857618f4302347f',
    '9bc09acf1': '118d936ffcae6832bcd5806cb106ab9e7da77a1a203c6301626dd99739810712',
    '9c72bafb6': '072883cbf19e836e4250db00cc18c9ad61ab74a816f12e437ed567bcb80ef001',
    '9cb0a5504': 'd88515e9303f6ac503072b53916321373ba0c597430ccd60efbbff4b4ef5c8f5',
    '9dcba0cab': '1a055b2fb5896df755d037a6a319eea24b9bfb7dc4b8bc32a421cde8e7e79cc6',
    '9e4780f8b': '309d9cd261ab031b1f33dc67b458ef7f90e9143f91dadfaa00c13e7db699af8a',
    '9e5f98ad5': '2d20fefc39f79ca915dd9827df82cab52d15a39b54c9e387d1b32d1cd2ec259d',
    '9efadd5cd': '833e27c4e6d313108e4f125191984685c905b81d90789fcd233050ab240efbfb',
    'a024116dc': '49665775e8bf101bba09c06cd347b44947b2249d8583e9f4f34199af3fa6f0db',
    'a06dbb0fd': 'bffb91e4231c6c3c7332e305e4acd1bba14fe893f0782af2251380800f4cd15b',
    'a18ff674f': 'f392c2ba2ebbdb004f2404a87e734c34e54e350466d9b731a7067233e6edc154',
    'a28a4ad8c': '058a3cbe186965aa0ae8d11b1b99c3b09b1e2b7befb9f891a9826c3a12512084',
    'a2d3f8588': '302fabdb84a481dd635a91fe3455b63f4d2d2171a3ea3ca3ffc74c1edd6873c3',
    'a2e7fd80f': 'aaaab55c5b8fca3db6fcd8c28d5f0bd3221e833451eff3fee705897f185c6924',
    'a3773af88': 'bcfa0b3cbd8de681b3788682855a9da9053fba1c1e2ced800a8fa179804a28f1',
    'a39887dc2': 'cbd69539d696aa23ae034eccf2a4f9cc936597c003f343df41cffaf54c48ea73',
    'a44b0ece4': '8df97569e3ea9e351f82420cf81425ca90430c49b6c358b05917ff2792b52696',
    'a5c8d5f5c': '659687c38d2e15d7472719a04131ed770053ea5421fc3885c4fcd8d48939b04d',
    'a65b1b9a6': '337c27378da9e1a3122f57fe1892206d185e7679afb9579a9b9cb6c11eb00962',
    'a666d9bcd': '7c732dc01b4a4b1f577c977bc6659a26d428049c129b9643383ca9faf114b8bd',
    'a673347b1': '19f0bb9fa1f03dfd0eae81f5ae3fe93f78f6c48d03f3682cdbadc9b7bb5c2f52',
    'a6c205e19': '5b2a4c04bd70c2e959a149d50f3917212a0754750a730e4cb0d4542cfcb9cbe9',
    'a7365ee55': '219b5260833d054e92e4349479917c4a75ccf36f0ce9ac786d29516460dc728f',
    'a7605a14e': '7ad985898295c0abd691c90fb9bd32c7776c826e76b0797fdae93c9c262c9d56',
    'a76360ece': 'ebde367aa2ba0b23e7378bbf915db06ded5d640a52d0385d44d0804168707d9e',
    'a77598408': '5eee34456b9a5cf15436b4782482a5f123d3a616fc319e03e726ca06883d7ecf',
    'a88930f1f': 'bcb9857e2f9f3c5da52def8bc75be6473f380a16d2b524d70baa9fb0a35a0362',
    'a8b61f66a': '87f2f3a918bb5196a4e37ef98405236607fd4a7c5f0e3032f52e24022c93ada2',
    'a8e1c8b37': 'd60da7114d4dd857c4e305c4e658e9b098ff8c58f6ec8e40a5444a11da6d2569',
    'a965d4117': '1ae8d94edabf15e7d182a5a1b9fce060048314fff8d982b55758a1db7a99a3db',
    'a9cd1638c': 'd8e307f30adadd04c8fe213da4b7b87658a6f6a74bc1006aa71d1329226b4657',
    'aa82bfd6a': '7f00f6da2b658954b43a6884dff5a04b66fc14bdf7516e6dd44fa5ce7729ee18',
    'ab23f5b74': '834ae5a6de591e97410aadc8bb73f53eb766ab19ff17f848d279863d58b1b69f',
    'ab4ff3cfa': '58c91d14058d5041daaca478f0510ad7d74cabbd7efae6617c5b56e7f91aa9fe',
    'ab81b2342': 'e0d7db0f1813ff10d6f33f61b46d8022592ea3428c1d35214337284d9f745f7c',
    'abcb46fc1': 'b2ad8069eb0dc75fcc4fdafe654b3b2cf168e560b7aa6d2d0d2093042189ad7a',
    'abec65871': 'cfdf28d5f7da467eeb66df15418d56246ee04c94acf5537266007ff84d41cb94',
    'ac47b0c64': 'b6fe4b980d797bd3e55dde07d511f5be4f95fe113b72ffffcd59251e777295cd',
    'ac6b1b3a3': '44525d277470db1b393cf4cfa5da03552fe47e41a008e81faea32219bc2daed6',
    'ad5ef99be': 'cd3faccfb74728f704c16845114f4e7697fbb5f25a340120d92a4ae6e7716974',
    'ae876f3ef': '5377dc1429cad56f35d3dd2a9eaa5972fac58e70f8ed5bea4733f0f8108bf9bf',
    'aec95891a': '262e7f87b23465d3989a3f8459845dab74dbae621105561c2ae41d9839a0d6f2',
    'aedca9e2e': '7c2644c5ac2937857f19cf21d3ecec0ee31a67f1a3acd7fa7ffb536ceb5f9307',
    'af32eb10c': 'f67d98b43cd561964962d98689039fb4411692d2fa79329a927a71b42bc5acf6',
    'af4f717aa': '0587453e8062ece921764a43e05952e365aeeddd1fcbfc30c826dbc6009a5d42',
    'af785ce1e': '9b8ba3baa9731fd13319c8671b4f7764542d4fb20d8ebf087f15f62cfef3ffe2',
    'afb9f53cd': 'cdc9eba5ae0eca41133ba8ed8bd06964da5c299203d5909f4a3f78057c6f487c',
    'afe608129': '297322cc429999847839593b3f949cf53860089e0a9df7615fdf347d6db60231',
    'affe1643d': 'da3a6bfa8f591f2fe3f26419e3f953c4e64b67a45de684032ad14a2f18bb66da',
    'b035aabf0': '633d2206e9e50415f8dccb7a96051c3f1a157bf35a77e49ff40d45664b54806b',
    'b0d4a70e0': '83562f1d446648467f653f4364252b19762b6eb89441f1b4e6d253b7c51344a4',
    'b1642dc6f': 'decf8a3ee7212b80e284b404178aa73086f9883adb204db8fafd77cdb708f765',
    'b22f456aa': '4455b2b0b7a7d8fd8935d38608d04f51227f04be18874ef08ea7cb9c54d99e62',
    'b28fb55cc': 'f44881207f87446d2510e035f32c94035b0dab5fcce9cd9a77dd2883ebba0c89',
    'b2d397a82': '21454b6ffc2313e3c11fb8508c5bd6a82172a4e4d549a4a5873968e9d495d03c',
    'b3822b5cd': '9925c0cebc4cb342bbd2a2696b88db7f847abae69e0267ceb4120193dad307a4',
    'b3bd71a4b': 'ca3bbc5774e0f08fa674031aeac4f569402a815f0ead42619cc232b39024c7ec',
    'b3ea991e1': '7b624e534baa8dd8eb06c92b6c1d1fd5cd6fd49c54c0db5f258fc5c6511ad8a3',
    'b4661eb6f': '1fbe464583de4671afa050918ab60ae7f30b0d74a8593facd53cdd1818e6c804',
    'b46dce732': '9eea105d740d6dc3a5cd31a6dd93f9e20ab690d45f8717626ee46c2c296de8a2',
    'b4d60b5a2': 'fadc3683ae9e1708d777972b4ad3df658a04b83c2472870a2b05873d6b7d9a3e',
    'b58605092': '9836d7c237229f80e57d20800c92f81b0b1b302b0d1d65c8b9aa387922bc9e30',
    'b61e9bd86': '75e6abdd32fe5dc676ff6f0686199b847d6bb155aa62e41b5cfc9bfc01ac39ff',
    'b61ff5016': 'f61423da262b4c8718f47bc8f23d6a7f19bb5f5d6ae400f1145c4333ef6adc66',
    'b6265c083': 'db9ab22e4a3f0db0c486183f2f7591382a449bd76ba8b36f634fa4581f9a5d9e',
    'b67b4d2e8': '7dc16ffb91e5e1d006b5cc6dc4501f56dcd16ead2751dcb9a74ed9aa853d3e32',
    'b6e406cd1': '2d9cfa0d57e14dbcc7e388883e78a09339b221372fc176b92efd763b3d9d2391',
    'b6eb131d8': 'c2e14534b866bdfa6a51acdb101b7815c8ded945765415721ba239d92a3ccfd9',
    'b72ba28ed': '654432320a1c009301a60e27121579b59670cd154a56d368c089a13d1a75f769',
    'b745b9e4d': '285716f751edd2e04550a2febed70ad9dfcb2612d6795de47d711c4195c45781',
    'b7c9166b6': 'f11905cead0740813dc3c613e713ef2c411cf0532ace9cc36beb15bf1f754d6f',
    'b7d03627f': 'd5ddb3fb270bd1925d88239cd688fbaaf437d62a9ae3f764143b7a19918a6e75',
    'b80ec5988': 'dd298d94ed62eb3a16e213a0aee89d89b83d1f674d7e77bcc8eb46561f9a06f0',
    'b838eaf1d': '7a4bdec9487a2ae7dd4033945504a8b9c80229d708774155b4cd06aa67c73c79',
    'b899eec9f': 'a9146239bdb1bf5963fc48db33a04a1c4ba6569c4e83ad01e3e5966358c1590f',
    'b8ddb6c73': '4a8a39185e7fb87eb1067d78e502de53fa2eed55dea959545b8b093d7d7a648d',
    'b96e8e1f5': 'edaf498fed0cb4baf7ca8ca859b52c76d48bb6865b3727654609a598d617a104',
    'b9bffd5e2': '1db59ebd14a58d25b0d6b0f50d63d503ad3aa29f8dca73829e4287d14bd03bed',
    'b9c9c2559': '330bcd6a33ede30e72d25b53863f9c53bed99ccb9311fea961412c915be7b010',
    'ba3404681': 'c4b8ff6b18d49bf13c9122f715c9cd9ee3646c9b53675637e87f2cb51e18b351',
    'bb3df77ab': 'd9a97ec852f6d44480e247b3ccb9e50983181a06f7b3ebab6b4879fd5a0ce46e',
    'bb6229e9c': 'e3b50bc4857f54288a3a47b5b77b2b91e25232b29906317cb351d9e044439dd3',
    'bc49e2888': 'cb118df36fd20fb5eb7e950eacff98a327284e971e60116edd06a8e5f2e6df36',
    'bce2fdc4d': '5a9f25976bd36dae316a28fee6bc67c98aada3bb242ddd1b3a4a1746a2b5da08',
    'bd20bc5c9': '100e4ec061b2619b5348b95033ca3cc3cbbe982a91b164b416f137ec80b07d4b',
    'bd53c4a79': 'b0ec7a2ff1d76187f94933b04ca065faafb5b76e07ec9d3bc45d734b016a347f',
    'bd9083329': 'a8c1d5bee194771ae7f56345574b80587190f0f017234f7b92b8347653e66f81',
    'be29a83ad': '608d27482e40383a1b3a7dccd47be7f8f158be5e127a21f3699bb4976c3c75cc',
    'bf2027ed3': 'd8aaf1954d28366f57890874285268d1afa4a41752f7f893dbd6feca721e877a',
    'bf4a133c9': '805b1f220f4c7b9777151e54bdb660984f9cf16a2740b5f3b9fea20dfee68499',
    'bfe0b9132': 'b02dd7b4624aa45f975c5f6515b88b24b8a40c5c1ebafd3b049e56e85c79fbf1',
    'c022ddc42': 'bfe45294382679096b0ab99ca68e07963348023fa0535db11a2554946511d290',
    'c15a563d8': 'd8c4c319d695098e45bc36430a272b61e86131f3871cb5b9d6899f1d11c79d12',
    'c223249af': '9f70e707eb3ae13837f111b388060a3422b661d71bb9278278dcd5eff9e71567',
    'c23c950a9': '67617dc67964e0c6802c2a1d83439854be06ccec037faf92fe43fe068ce33104',
    'c25cf1f92': '46f11d77edacc79e90431e8080a422dcb4069e38ff8aa4fef070f9a5b04dd475',
    'c26cd234a': 'f2b374d29779ea486f2b3691c537f0e8f40a2140016c2680928cf4b4b1265e43',
    'c35347675': '46b5af06fa60b87060395b8625b0952485e195b371ed7693e4ff5a749c843d9c',
    'c354a54ac': 'fcd8d2a3cc9ef93e4f65533c19e8111c963c6ea69745513be3a395aad43f5991',
    'c35743243': '435bae57eafd4fe6e9c0d435f07ff176b738566fd5d80ade5e567d53760d8cea',
    'c3a8a41cd': '7f43cd59b8e29c3480b593b177c4f3ae97b2c4e62d88aa8b92efd1dd46c246b1',
    'c446944d6': '58969c8c13995918b6abef779e1cefcf6b59fab1402a3a42a52ffbefca1d47d2',
    'c4731ac9e': '4790cd2aa051df57b6985f0abf567360086b45bf29bc72248f0a0d0c7e7aecb7',
    'c4ffde64b': '96d392345b77eae66914c28932de1ac94ff8bf5119d3050325cb58a0384b7540',
    'c580831f7': '2244f96c20b142c6100ea15f501a2713223258eabc06508b7e5f97199a482cc3',
    'c672ca0b0': '032302c578aa9694275c224ebf77d707ce6f6d199439cf41601d7ab68595543c',
    'c6a299f4d': 'c7a6343f1c63de3ed2f7e649d56c2e23a3c31d469d2cb75c877beed4b6dff3e5',
    'c6ca898c2': '0f2d74671e5e32aa3a060fa86d552695816ce5a44bc65da3f160b6508a37d487',
    'c7532e8e6': '1d83b3414a67c261aceb6a86341a6e6a2a5662500d8d9a57a9c060c79329f918',
    'c76f1c69b': 'd2331156cf1644ec4e2b2055d1d44aa79aa178fc3e74b158a5e02caa7fe97655',
    'c78edc091': '207b1ae4e3c475db33cb81d5c84ee3d5d649bee86baea0c3a54baa3538ae8699',
    'c7ba07095': 'fc98082ce3d5e343735f688d529e1c2f5ad0b3c98685d87eb942321436aaf4af',
    'c7c9b4e72': 'cd36559ed6fb64eff1db2f9b35c1573b3bc088d8252831032d6215fa1a4dd30b',
    'c8237c2e6': '17695c7f421e725d9e1c194596af10ad404123e4c0e8f5b96e67046c31892298',
    'c85eedb4c': '6236b7f28e52281411a46ec80e2641988dd04b1ac5acb654c06d4dff3d6c9580',
    'c88635e09': 'f4cd9a2dc3db8b074e088283ad69fa28a48d50c5b2562e12fbcb87bbb1ca7729',
    'c8927c5e1': '9a1f6bf27a68b23f51be6103b4b10cb9532a1f2202d3ccb0ab9015539bdf004c',
    'c8b81529c': 'a612793aed0a13f1e9fe0d65f2d77e0c776ee7bf6c023ef8785aa36bc97bafee',
    'c8e5e50ec': '5b66bf3849511ad8916d356d19bc579a7ef17b71f73c68a154e4f32c2b28aa18',
    'c988834c9': 'f3b1012333ec169260dd0a8d1940a7a486f87b0335a7e5928145c6127e21aafc',
    'c9d60e075': '457f77b51fe63a5227d0e90015e3da319b934233bae4ea5854fde3586b414a03',
    'ca08214f1': 'b120fb75b1dbe1f5ac21d1a2137267dc7324fcaf7021e1931a65f009eb4114ef',
    'ca2b57db7': 'df2272c7cbd9ce29366e89df7225a4695b4a00097d33226ef3ab80f264f9aec4',
    'ca56c6d41': 'e07cd29c5a9f6b26d9276a290d7d6f9a2a54c443c1be81721014a338e3e4c7e3',
    'ca664da7d': '4bc0900de1e1286125ca19ab3b8ada3eaa374d544797e8350256b255136e6ffa',
    'caa7d043e': 'beb78902d8c9ca8145a87e8b8f485e75328f9709202b35bbccf74133256df4aa',
    'cacd820db': 'a5d099b73ce330055b20c0d2ae2ae650d052127c64728df282fbe456cef92547',
    'cb256df87': 'd316022e8a4483c922a9f695b58cda4d9618f58990bc5723048e0e5805e9005e',
    'cbbc58a4c': 'ac655990c3977ad42ce92defeb97a62e97b059e10a83e7f38653f03f2360dedb',
    'cc2bbb768': '1ece74cc55bf7dabb3b82f583b98e8121b7ae3589b2537a32467e6b0120030e0',
    'ccb8097e3': '548cff9d63c6b18205f340b05da8c64a89f71b04cba980596682b08e35a5e259',
    'ccf2bea3e': 'c11126ef9b02cae4ffcbbfc6e8eca82fe127c1cb8bd032bd33d3a275e0f582bf',
    'cdd3bc83b': 'c5204852dff267cec6a2a64d8d402cc63c2e39151bb0dbaeaf2656c8b392c885',
    'ce2e2fe79': '0221b422fc18efc07a9ca01237cd8277b988f341c6d069a433a703264d31c404',
    'ce69098f5': 'f41abae4902c9107a3c6eed5d71e643b8180d398071e87376ab9a6f82cc94fd5',
    'cecd9f0c7': '31457ee2d36f62957d57fd048ca59f36cf33392e9e833386ab78a9484ac3189d',
    'cf483de89': '61f92c37fef1f73dc7e00ceaaee1a9f99f887a0ad313d687841aba2d64d99af5',
    'd011146e3': '22d9d54fc333709eb4f9810a72d2c793a38441a9b12a5cc76bbe573881f0859d',
    'd03043abb': '2699a6797b78695782bd7e9a13e7b6a10aa570ff4eba5fb74d871f5c523a5105',
    'd0912a485': 'c3f964cdce9b14a1f0e51a82130d847ac21fff2273174dbc855637a3a820ce65',
    'd0ca0d7a0': '36ebb9e39f8b045d4959bf5cf2a3848e1fcd99b876b0be4e44c1c846a50403eb',
    'd17ddb23c': '2c30d42ecc5c59b1a476ab38651111bdf7c2a37fbf7259106733c548d89373a0',
    'd17efcdba': 'd60132ebde25476474df27b6628719b97e6110cdfce63df16db32fe9c87165a0',
    'd19c0b3c9': 'bbb81c73bfdb3427a9283b49208b6cf77847a5c0d77fc82d8f6cdabcd2de4898',
    'd1e5fae98': '6ab4a90eab8cf78fc051479a17db2de34d3e6aa034310c925f8825770803ffc1',
    'd1ea44cc7': '26dc4a20551ac48f1f34d66a5ded4d30ed78cb47f08e1002af534843566420dd',
    'd1ff5f449': 'b115f9935a558e0640f9518664996fabeac8fd451fea9b9062287ea48d4cdfb5',
    'd22905ef2': '9c66fa4d4214d5415306329c7bb02401432eee95d1dccd96ac04a4a1581036b7',
    'd2f56e67a': 'b92032f1845ae0c4d82d3abd1f622942e21522f58a1077f57fcdc1fb21007b52',
    'd36c62b8a': '5bdc7ad69250f3b5a3ad51b8d3078b2a4097cd7539b496dd7323c49c058bbe2c',
    'd39457b35': '654dcc94a7b728d5737ffb64fe2de9e41e8094dd640eb356d057aa6703bcf293',
    'd3a6e23a3': '05467c888e48f22c361696b26f9b87ab188acb5196be1a9bb866023e008c30ae',
    'd474f39c8': '0c7a285e4bb5b110926f46d36d45a43f8ab35e0c5032c4173d8203eaba34fb00',
    'd4949ae24': 'dcab674c75f4bf0f7835cc1969b02063ad7b84c7974827dcbfa653eff1559059',
    'd54ece3f3': 'b8e08c71a47652b34ff9c5451bef54f6a55f660906b3cd37aced3db3f0995878',
    'd557e8635': '8f1dab145cd0e4a15901cbe54bdfd7954e1cfd998c410c1ec3732b7369e97f12',
    'd5bc2cad4': '7fd7ea21340074299a2155da0010d1327f02374b95c0ccadfb51f4037ce32e1a',
    'd5cc960f7': '5301582b10241eccf15d3213d01e3330687b4f01dd88ba7ca0ca2f4d7b36feae',
    'd606fe447': '8219dba35139ac458f0393e42780d98c09f956d27cbef7d7ded5e654603f23d4',
    'd60e832a5': '121a792d3267c451d06ea454f126877cdf59a4149e9543679ea5e2dab94a8186',
    'd63d5214a': '392bd56a388d118ae5ff141a75fa0cb3baba29ebbec1c4b77e2157df0d4b4d52',
    'd6cd9fa5f': '5479cbeb1cf55e5755feccc411f8a4fecdf528640d19934bbd5ced9accb0692f',
    'd78825345': '259b66e6a5f9c5536a12a7c54c1839cf985043265f87c89812fe9038d18381f4',
    'd8d946295': '1e9102a8910cacaa05bde65ee6bb807196836ef268675417ef6e9c70bf9ba552',
    'd8e38159a': 'a2ac6fc7a135538fbbee9f4a8245fb4013b2e91636cc9f70fdb9eef7925ef833',
    'd950bfe39': '0fceed1a48d343256f6172b02b303fd40a5731fb668a0f806972b0ec3457ccdf',
    'da0e870d9': '97cea5a83d00ae2fb067acc35088c3191dbdfafdb94378d6a85b29df6c4b66d9',
    'da3cfa3d4': 'd2df4bab2592cafe8d536e3e90b2e7eb364715a82cd0dff26b5b7a9a0881edc1',
    'dc620a761': '848c321b09f5cea6ef7fbf143812218d9dee865291b6afc797629d330d86933b',
    'dc75973ed': '8c87a9ae9b4bc30e9543ade9ba504116c8bc48d50703244dc2cce6efaa3ea099',
    'dc8e4942d': '50b9906ef308b4be23f396f2a9252eee2f3ed90bf69f6c83098890793c804941',
    'dd5dd0234': 'ff586d6ea0360d7fe7ea882f9e405b7bc42658744d2198ca4c0011f48a387d17',
    'dd8ad50bd': 'c9cfb4dcbbb228a58e4a49fdb46dda4ebe101f921b2a84245f5ae3e57406fb9e',
    'ddd60739d': '22a5dd5ab7285ca3834ea7fc5a30503cdbc7c47da721182402a0d7792d90efde',
    'de4e0eecf': '904d77e5cc477b1ea29dc5adbc80d64c24e901239147e40c670590f6f407da64',
    'decc5459b': 'a5b8d1eea74f1eca7170ed7976ca9b26d32406084187978daa2abb92b25cf2fe',
    'df1ba5253': '2de17579ab4924cc3175b2f85ca9836b460ec52b41b9477f61fa230a03adab8c',
    'dfc6a2bcc': 'bddd978e41d003e0c338f63c8e65e71e38a5f6e1cd72b7abf5b650eea75e6702',
    'dffff7fe4': 'a7c7748bbbfec6404bad551d2da42ec28f5589a63d4cf7b24c59bd4a95d682a6',
    'e011bf67e': '6ea9f20d52fb3b4850d2c8e1c739dd741e74c5c8a4fe22533ab445269f0fbd8b',
    'e05767cda': '4af19e9d1778910bc05f573cb621ea08636d1568ae17c66b933a06ee99e48467',
    'e089dca60': 'a6ebb387fa672b949e2cecf1aed4df7548a4d2bbdde3cceb0ac385b650dcd456',
    'e14f4de23': '3a1b6bff32c5d2dd4ccfaa46a84a7ff770d68f24969d003f43e0626abd3245e8',
    'e16e3b2c6': '0956eea0c3206324c9228a244f19ed66ee3c1b5bd6ed44d0317efdf1f08fdb8d',
    'e1f6c91bb': '20c248c0ae53c915b8cf5f71e032c1084d9fe06138004397e0f867bf77a5f682',
    'e1fe7d33c': '40fe12c75914393e20618c7730d2906f2445cae618516d9776ad4be55465f54b',
    'e2ac8324c': '59e2bc23d50c582239b2c42825256147fed33270c01702c47adaf01e273f7eb7',
    'e30cd9270': '3ec5d9d3c242defef02cb0a65f3fd54f0c2a8d50b992285ce358da2f7f4a7b82',
    'e321272b3': '990cf51ef4fa9490e1afc32bfff7dfd05c2d83355971c23d8646d10c503e08e0',
    'e46eadfe8': 'b4d1da2f4dfba84703596a7bb3092e3ff3d18da127f8a0e2c33d413d2de11310',
    'e4cc39cbf': '1b2917c719220c8f2b5a27377bdf6ea052648cdadbd02efac9e326ecd605a7ed',
    'e5c0a16c5': 'e8d6ba177c74a48c99e82282551432c7db81218860827a575d1f4993969e5424',
    'e60108508': 'ec5c22a7f5d646d91db67369c6654e799beb15d1ab37fea04a68162913815335',
    'e6277ce7c': 'ca18bdef6716e96918c1dfb113eb05d74222d4c20815c61cb4f8d225d6286f65',
    'e6989f6c7': '9629e3202c1a3eb6b3cde523a531b9904a90988f352751e681c9e0e431185cbe',
    'e6af62317': '523ae7d38ad6aaee85eb1c5ad922bed847e550d11ac0147064f7a559c5daa434',
    'e6b5e296d': 'a0534fda946d3f9729941f30881d630bc3cf899ec7deda8aa8a22f75f8e7d09f',
    'e7527a057': 'e4ca83922a879e766e3bbcf07b89ddf0dcbcb99fdddfcad620d9d2f895a8127e',
    'e84d2bbd1': '41db85469edc1ebdea31144974fd1a4bfd3e7f4892f93572e6f09e6b1f87d1cb',
    'e87d43a13': '2df226fd62d407a2c8cae929d0dce4da4608bfa33f447041b1e201a3b038fa2a',
    'e8a274b57': '57a08ba2faaddb7bc3908819a7ae9a4f9692c5088fe0dbeefaee496e49cabf4b',
    'e960b67ea': 'cbe8ead1005a5c36d7e890de897efdf5e5535618fc872c5969ac2aa6e56f5f30',
    'e97102590': 'b45575caf15e0235bc22c0bfd19493464af2fd3934372d5869f4552291a3cba4',
    'e9dc0cf8f': 'b67f1d0480f50bec466beb432d8d40d742ff83e8a020b0388a38447db365a8b3',
    'ea89a7320': '49a18eabeff93416ec6ffcc23669942d3d77cb303979389adedf6312d10c7b86',
    'eab32c67a': '06f1de1dc65c51d51f7d174b905b27c04a02352bfce8e3e507f8ef1a793765cd',
    'eae9f9037': 'f443f93fe17da1c34a5e3db6b10d50417c9698f45ca404550c0e2d43fd0e88ba',
    'eb42212f3': 'a6368f2907d0a8e077f5e0aaa8222538a33088a32d86e94743524cad7a247086',
    'eb69f5522': 'ffc2b8a46a96741b6c377aaf6b12982c03aab5dccb9e8ffb3dfc129749281df6',
    'ebbc11f78': '55793e87610d89b888d04500231168b0922a1a06967d49f03f9c0d8abb36fdb5',
    'ec3d2ced1': 'ef4afc36566a2570415a9e4efe9aa468f88ca5614e71c57b57f953382b06ad67',
    'ec8640868': '8eacc36c62e36aab7dd03b6b5170987709e1632e71f4b16bf116b04550832142',
    'eea7d700c': 'a4d15985896b4d78d788cfeb2c5a4d56ae916d92ac97e511c94699969373118f',
    'eefb4a7fa': 'fd6c73a8bf571b246839118ff6c067dd6b6f5aa9aafaa58bad6c317b1ac197ae',
    'ef274ecdc': 'c3f76554975e028af5fffb590696836fe33d2d9a5cb7cebf892a4692355d030f',
    'efc4f0374': '14428dbec8f9376de3f3381dd101655dce6ee486427f51d0d2c135576636e82f',
    'f0508910d': '3dade7bfa700afaa11b0624442dee203bbf6b27e40589c081dfd4f35db80e43c',
    'f05647ed3': 'f708355f939f287fa59b5dc2826484e023ae60d984477e288e3439244e466da8',
    'f0b1b4fa3': '8b781160fa6a24bd060a5db97b8a47a0fb5abb6c23707dbec506cc5ab1c9cab9',
    'f1016e0ea': '7d659dc7f04dd34922571b1f5448924a3078925c01a38758377e9cc3e36f00b4',
    'f10db69eb': '9ddb6b40418fa005f0d72d5f331ceab83949e31e16f0240056cf11c7645a48a6',
    'f20baf558': '7706af64404fdfa9003da27b7b71b65c75785e105353f680443bed0fbf707b73',
    'f21d6d68c': '0c77e84c12fd2b9a6c034a36212c0a6def39b128c93efd290200fcdbfd2598e4',
    'f24698e88': '4c58ad90de3802f0d0cba221d7ae444a80930d5865327676ff5bf53312cec069',
    'f2c0e95ec': '5c66ee0c88e0ab40347bcfc958abe984aefb629865729aa0b9a360a936affa43',
    'f355c9fe2': 'cc1fddd1843f3ec3ea76834f01ff680d585f6930550f4eae23107cbaac1aa42b',
    'f3bc43304': '37a2c3b68bf4d92f6c3d5eb63aa98ca882c00394ac507164df7c8d8a98e828f2',
    'f415daf80': '90bfb444d84fe76e46c04d4d451c69365ea3b14650c1a29740776c9063eed13b',
    'f54dedb60': '80bebd77f36d7d7f5b18db86294838ec71023bb716935f980750c635b4f804dd',
    'f54ff03a2': '47a9e9e38501a558a5ccd81c625bf0168a4c39c8d412cb6f8717555829766e55',
    'f5607392f': 'fb9cb6be29e484f7c5600408bdf97a5cfeeab17501d4aba5cd56d595cd8a717f',
    'f5b320e47': '3faa2fbde63dda82d05fc4ce5e96e564403fab5c9902dcdbfc8cd48cdf931429',
    'f74ede365': '0ff56ec8e522984fb912aaf7d8b05947fd1dd38c09058a5effa9d005235e4641',
    'f84383b38': '2f76dc0d8360c6572b75b599810345a4507e6e1d8743353003bd498ffc69f197',
    'f8d155138': '2653fba82a191250374a3576fba5c8a773a754b9bdcdcd205a29cfd4172311c1',
    'f91e92fd4': '606ec3eca99d0f95fbc6a1d8801067254291ff0e14a31cea55a4cf6d26874518',
    'f98d398d0': '10f3406b25790b6f94bc5e689fe82a2e9a2ef3ae6c9e0f1f61680a79288ed133',
    'f9ece438e': '86c4323abba690367198a37f6680cf2475cf2b3ca9dbe38673abc1a784cb32ec',
    'fa060c1f8': '3d52fff084abeffda47e748c6e664d2ea4df74f1a472903c4f08fbc705b22563',
    'fa9b3ea2d': '9632a2419d36701a924eb51afef98104cbd18fd565c633b2d8b8bfc01dea47f6',
    'fb52ca9b2': '6d97eacc8a9ebc086644d1d5cd1ce5576d12fd4304fdabddfe0e542a3cd642bb',
    'fb6e04c0c': '0ba7c0ddcf6a23165a61663c4effbfd78cbef912ac07f2e729506a478cd6571e',
    'fb804ab78': 'ec9071b8106e1d80cc296c7795f39136f73e87e81c29fa2b011399e4657d9ba1',
    'fd95e2677': 'd27dc70933ef309242a8af5fc350bca9fbebd63a89c8c8c1354bde4292e62943',
    'fe1266f43': '839229c959fc9884e2993a2fe1a5556223fc66d054ec4172ae71a8875fff7a24',
    'fe133ccb4': '419e12c6b2444e93eab2eedaf1ccf857e3a7367927d5c74bdea827c88e218234',
    'fe2a54e65': '368a2497d430254b0468372e3326d94f9f243bbf9477a48ba13c1c56fad533a6',
    'fe342a5bf': '3296b3bb77e7fe8777e508933809090ae1ae2b395f9d63fd87831d08f6407144',
    'fe7ecb1f4': '988d7fff4191f2f0e4e90d128f1399f042c5e5e1574570788feacc15ac5c38eb',
    'fe8876602': '996d27c610aa1ababa98533326e9182e0e79415e3da3efd0d49d51590f37c514',
    'feb78a8de': '329a6b316c3a9417aa24ae3567fa1645289e4465f72bdfa84b1b072694bb1e11',
    'ff5c9104a': '3a599b2040f23f5724c15c9ec8f9ba2e0aff9d97861d042880d0435fd5218f21',
    'ff86547fb': 'e2b5fe0dfef3915054b55b9d38140ef651566a3f327b5e50f1e5c9d97b15f424',
    'ffb445410': '50275982f2cb0479d296ef36f240bf131c8b9912cf7549709693265bb6902ecb',
}
from pathlib import Path

spec = json.loads('{"notebook": "yutarotaro/wheat-benchmark-d2-tiles-validation", "source_notebook": "wheat-benchmark-d2-tiles-validation.ipynb", "run_label": "d2-mbv3-wbf045-d2x2", "description": "固定validationの画像でEfficientDet-D2の4タイル候補を採取する。枠の評価・採用判定はローカルの固定評価で実施する。比較元は提出58370638。", "wheel_dataset": "yutarotaro/wheat-offline-inference-wheels", "wheels": {"effdet-0.4.1-py3-none-any.whl": "10889a226228d515c948e3fcf811e64c0d78d7aa94823a300045653b9c284cb7", "ensemble_boxes-1.0.9-py3-none-any.whl": "f095f34d28034213791b8793b9de4d8ba6e40e6a2a6af02336b84c485d853d6a", "timm-1.0.22-py3-none-any.whl": "888981753e65cbaacfc07494370138b1700a27b1f0af587f4f9b47bc024161d0"}, "models": [{"trial_id": "8fc98f3e68e8441a-0001", "notebook": "yutarotaro/wheat-train-mbv3-12e-640-hflip-ef473611c538/1", "sha256": "5e2e80f444bd82b2d892eef710836f3fb6cf45d5028ae8bbd97bd062bbbfe1df", "config": {"image_size": 640, "run_label": "mbv3-12e-640-hflip", "epochs": 12, "batch_size": 4, "learning_rate": 0.0001, "horizontal_flip": 0.5, "vertical_flip": 0, "color_jitter": 0, "validation_loss": true, "confidence": 0.3, "nms": 0.5}}, {"trial_id": "d31623fdd5114e49-0001", "notebook": "yutarotaro/wheat-train-effdet-d2-6e-768-b995f1851fd0/1", "sha256": "3d565f728235bb868541a55e6bae1248dc2a71167d480dbce3dadf3f83dc5bf3", "config": {"detector": "efficientdet-d2", "image_size": 768, "run_label": "effdet-d2-6e-768", "epochs": 6, "batch_size": 4, "learning_rate": 0.0001, "horizontal_flip": 0.5, "vertical_flip": 0, "color_jitter": 0, "mosaic": 0, "validation_loss": true, "confidence": 0.3, "nms": 0.5}}], "weights": [0.3333333333333333, 0.6666666666666666], "confidence": 0.4, "iou": 0.45, "validation_score": 0.5872414935110463, "holdout_score": 0.5570303071173651, "partition": "validation", "tiles": {"model_index": 1, "window_fraction": 0.65, "seam_margin": 0.02, "max_area": 0.008, "iou": 0.5, "rescue_iou": 0.3, "rescue_confidence": 0.5}, "split_sha256": "53da7a99e83694f2d0f706a19bb7f9675f23d29b767126c4dc16755aa777710f", "csv_sha256": "df5738646745fc477cb9c367a7e4b077bd868797b429e6d81fd4c5005ec0fc70"}')
spec['images'] = images
Path("submission-spec.json").write_text(json.dumps(spec))
wheel_paths = []
for name, expected in spec["wheels"].items():
    (wheel,) = Path("/kaggle/input").rglob(name)
    with wheel.open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != expected:
            raise ValueError(f"Wheel checksum mismatch: {name}")
    wheel_paths.append(str(wheel))
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                "--force-reinstall", *wheel_paths], check=True)


## data.py

In [ ]:
%%writefile data.py
"""Validate annotations and freeze a split by acquisition source."""

import argparse
import csv
import hashlib
import itertools
import json
import math
import random
from collections import Counter
from pathlib import Path

from PIL import Image


def sha(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def load_dataset(root):
    images = {}
    with (root / "train.csv").open(newline="") as handle:
        for row in csv.DictReader(handle):
            image_id, source = row["image_id"], row["source"]
            if not image_id or not source or Path(image_id).name != image_id:
                raise ValueError(f"Invalid image id or source: {image_id!r}, {source!r}")
            width, height = int(row["width"]), int(row["height"])
            if min(width, height) <= 0:
                raise ValueError(f"Invalid image dimensions: {image_id}")
            x, y, w, h = map(float, json.loads(row["bbox"]))
            if not all(map(math.isfinite, [x, y, w, h])) or min(x, y) < 0 or min(w, h) <= 0:
                raise ValueError(f"Invalid bounding box: {image_id}")
            if x + w > width + 1e-6 or y + h > height + 1e-6:
                raise ValueError(f"Out-of-bounds annotation: {image_id}")
            record = images.setdefault(image_id, dict(source=source, width=width, height=height, boxes=[]))
            if (record["source"], record["width"], record["height"]) != (source, width, height):
                raise ValueError(f"Inconsistent metadata: {image_id}")
            record["boxes"].append([x, y, min(x + w, width), min(y + h, height)])

    paths = {p.stem: p for p in (root / "train").glob("*.jpg")}
    missing = images.keys() - paths.keys()
    if missing:
        raise ValueError(f"Missing annotated images: {sorted(missing)[:5]}")
    unlabelled = sorted(paths.keys() - images.keys())
    if unlabelled:
        print(f"Excluding {len(unlabelled)} images with no source/annotation in train.csv.")
    checksums = {}
    for image_id, record in sorted(images.items()):
        path = paths[image_id]
        with Image.open(path) as im:
            if im.size != (record["width"], record["height"]):
                raise ValueError(f"Unexpected image dimensions: {image_id}")
        checksums[image_id] = sha(path)
    return images, checksums


def split_dataset(images, checksums, seed=42):
    parent = {r["source"]: r["source"] for r in images.values()}

    def group(s):
        while s != parent[s]:
            s = parent[s]
        return s

    seen = {}
    for image_id in sorted(images):
        source = images[image_id]["source"]
        previous = seen.setdefault(checksums[image_id], source)
        parent[group(source)] = group(previous)

    sizes = Counter(group(row["source"]) for row in images.values())
    groups = sorted(sizes)
    if len(groups) < 3:
        raise ValueError("Need at least three independent source groups.")
    rng = random.Random(seed)
    rng.shuffle(groups)
    target = (0.6, 0.2, 0.2)
    if len(groups) <= 10:
        assignments = itertools.product(range(3), repeat=len(groups))
    else:
        assignments = (rng.choices(range(3), target, k=len(groups)) for _ in range(10_000))
    winner, error = None, math.inf
    for assignment in assignments:
        if set(assignment) != {0, 1, 2}:
            continue
        counts = [sum(sizes[g] for g, part in zip(groups, assignment) if part == i) for i in range(3)]
        if counts[0] < max(counts[1:]):
            continue
        loss = sum((n / len(images) - goal) ** 2 for n, goal in zip(counts, target))
        if loss < error:
            winner, error = assignment, loss
    if winner is None:
        raise ValueError("No valid source-group split.")
    labels = {
        g: ("train", "validation", "holdout")[part] for g, part in zip(groups, winner)}
    partitions = {part: [] for part in ("train", "validation", "holdout")}
    for image_id, row in sorted(images.items()):
        partitions[labels[group(row["source"])]].append(image_id)
    return partitions


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=Path, default=Path("data/raw"))
    parser.add_argument("--output", type=Path, default=Path("data/prepared/split.json"))
    args = parser.parse_args()
    if args.output.exists():
        parser.error("The split is already frozen; use a different output path to create a new evaluation cohort.")
    images, hashes = load_dataset(args.data)
    split = split_dataset(images, hashes)
    args.output.parent.mkdir(parents=True, exist_ok=True)
    payload = {"seed": 42, "csv_sha256": sha(args.data / "train.csv"), "images": hashes, "partitions": split}
    with args.output.open("x") as handle:
        json.dump(payload, handle, sort_keys=True, indent=2)
    for part, ids in split.items():
        print(part, len(ids), dict(Counter(images[i]["source"] for i in ids)))


if __name__ == "__main__":
    main()


## metric.py

In [ ]:
%%writefile metric.py
"""Image-averaged wheat precision over IoU thresholds 0.50–0.75."""

import math

THRESHOLDS = tuple(n / 100 for n in range(50, 76, 5))


def area(box):
    return (box[2] - box[0]) * (box[3] - box[1])


def iou(a, b):
    overlap = max(0, min(a[2], b[2]) - max(a[0], b[0])) * max(0, min(a[3], b[3]) - max(a[1], b[1]))
    return overlap / (area(a) + area(b) - overlap)


def validate_predictions(rows, width, height):
    if not isinstance(rows, list):
        raise ValueError("Predictions must be a list.")
    for row in rows:
        if not isinstance(row, list) or len(row) != 5 or not all(
            isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x) for x in row
        ):
            raise ValueError(f"Expected finite [confidence,x1,y1,x2,y2]: {row!r}")
        confidence, x1, y1, x2, y2 = row
        if not (0 <= confidence <= 1 and 0 <= x1 < x2 <= width and 0 <= y1 < y2 <= height):
            raise ValueError(f"Invalid prediction: {row!r}")


def match(actual, predicted, threshold):
    remaining = set(range(len(actual)))
    hits, extras, duplicates = [], [], []
    for j in sorted(range(len(predicted)), key=lambda n: -predicted[n][0]):
        box = predicted[j][1:]
        overlaps = [iou(box, truth) for truth in actual]
        i = max(remaining, key=lambda k: (overlaps[k], -k), default=None)
        if i is not None and overlaps[i] > threshold:
            remaining.remove(i)
            hits.append(j)
        elif max(overlaps, default=0) > threshold:
            duplicates.append(j)
        else:
            extras.append(j)
    return hits, extras, duplicates, sorted(remaining)


def image_score(actual, predicted):
    if not actual and not predicted:
        return None
    scores = []
    for threshold in THRESHOLDS:
        hits, extra, duplicates, misses = match(actual, predicted, threshold)
        scores.append(len(hits) / (len(hits) + len(extra) + len(duplicates) + len(misses)))
    return sum(scores) / len(scores)


## fusion.py

In [ ]:
%%writefile fusion.py
"""Weighted Boxes Fusion for validated wheat detections in image coordinates."""

from ensemble_boxes.ensemble_boxes_wbf import weighted_boxes_fusion

from metric import validate_predictions


def fuse(rows, width, height, weights, threshold):
    scale = (width, height, width, height)
    for guesses in rows:
        validate_predictions(guesses, width, height)
    # Zero-confidence boxes carry no weight when averaging positions.
    rows = [[box for box in guesses if box[0] > 0] for guesses in rows]
    boxes, scores, _ = weighted_boxes_fusion(
        [[[value / size for value, size in zip(box[1:], scale)] for box in guesses] for guesses in rows],
        [[box[0] for box in guesses] for guesses in rows],
        [[1] * len(guesses) for guesses in rows],
        weights=weights, iou_thr=threshold, conf_type="box_and_model_avg",
    )
    # The library uses float32 cluster averages, which can round above 1.
    return [[min(float(conf), 1.0), *(min(float(value), 1.0) * size for value, size in zip(box, scale))]
            for box, conf in zip(boxes, scores)]


## model.py

In [ ]:
%%writefile model.py
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

import torch
from torch import nn
from torchvision.models.detection import (
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    fasterrcnn_mobilenet_v3_large_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms.functional import normalize, resize

EFFDET_PACKAGES = {"effdet": "0.4.1", "timm": "1.0.22"}


def prepare_efficientdet():
    missing = []
    for name, expected in EFFDET_PACKAGES.items():
        try:
            installed = version(name)
        except PackageNotFoundError:
            installed = None
        if installed != expected:
            missing.append(f"{name}=={expected}")
    if missing:
        if os.environ.get("ML_LOOP_REMOTE") != "kaggle":
            raise RuntimeError("Install the wheat group with uv sync --group wheat.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                        *missing], check=True)


class EfficientWheat(nn.Module):
    def __init__(self, config, pretrained):
        super().__init__()
        prepare_efficientdet()
        from effdet import DetBenchPredict, DetBenchTrain, create_model

        self.size = config["image_size"]
        self.confidence = config["confidence"]
        if config["nms"] != 0.5:
            raise ValueError("EfficientDet uses NMS at IoU 0.50.")
        if self.size % 128:
            raise ValueError("EfficientDet image size must be a multiple of 128.")
        network = create_model("tf_efficientdet_d2", num_classes=1, pretrained=pretrained,
                               pretrained_backbone=False, image_size=(self.size, self.size),
                               max_det_per_image=300)
        self.loss_bench = DetBenchTrain(network)
        self.predict_bench = DetBenchPredict(network)

    def forward(self, images, labels=None):
        shapes = [image.shape[-2:] for image in images]
        inputs = torch.stack([
            normalize(resize(image, [self.size, self.size], antialias=True),
                      [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            for image in images
        ])
        if labels is not None:
            boxes = []
            for label, (h, w) in zip(labels, shapes, strict=True):
                gt = label["boxes"][:, [1, 0, 3, 2]]
                boxes.append(gt * gt.new_tensor([self.size / h, self.size / w] * 2))
            info = {
                "bbox": boxes, "cls": [label["labels"] for label in labels],
                "img_scale": inputs.new_ones(len(images)),
                "img_size": inputs.new_full((len(images), 2), self.size),
            }
            return {"loss": self.loss_bench(inputs, info)["loss"]}

        detections = self.predict_bench(inputs)
        outputs = []
        for prediction, (h, w) in zip(detections, shapes, strict=True):
            if not torch.isfinite(prediction).all():
                raise ValueError("Non-finite EfficientDet prediction.")
            scores = prediction[:, 4]
            boxes = prediction[:, :4].clamp(0, self.size)
            boxes = boxes * boxes.new_tensor([w / self.size, h / self.size] * 2)
            keep = (scores >= self.confidence) & (boxes[:, 2:] > boxes[:, :2]).all(dim=1)
            outputs.append({"boxes": boxes[keep], "scores": scores[keep]})
        return outputs


def create_model(config, pretrained=True):
    detector = config.get("detector", "fasterrcnn-mobilenet-v3")
    if detector == "efficientdet-d2":
        return EfficientWheat(config, pretrained)
    if detector != "fasterrcnn-mobilenet-v3":
        raise ValueError(f"Unknown detector: {detector}")
    weights = FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1 if pretrained else None
    model = fasterrcnn_mobilenet_v3_large_fpn(
        weights=weights, weights_backbone=None, min_size=config["image_size"],
        max_size=config["image_size"], box_score_thresh=config["confidence"],
        box_nms_thresh=config["nms"], box_detections_per_img=300,
    )
    features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(features, 2)
    return model


## proposals.py

In [ ]:
%%writefile proposals.py
"""Full-image and overlapping tile proposals for wheat inference."""

import math

import numpy as np

from fusion import fuse
from metric import validate_predictions


def tile_predictions(model, image, spec, batch_size=4):
    height, width = image.shape[-2:]
    window = spec["window_fraction"]
    margin = spec["seam_margin"]
    if not 0.5 < window < 1 or not 0 <= margin < window - 0.5:
        raise ValueError("Expected overlapping tiles with a smaller seam margin.")
    tw, th = math.ceil(window * width), math.ceil(window * height)
    origins = [(x, y) for y in (0, height - th) for x in (0, width - tw)]
    rows = []
    for offset in range(0, len(origins), batch_size):
        positions = origins[offset:offset + batch_size]
        detections = model([image[:, y:y + th, x:x + tw] for x, y in positions])
        for (x, y), result in zip(positions, detections, strict=True):
            for score, (left, top, right, bottom) in zip(
                result["scores"].tolist(), result["boxes"].tolist(), strict=True
            ):
                if ((x > 0 and left < margin * width) or
                    (y > 0 and top < margin * height) or
                    (x + tw < width and right > tw - margin * width) or
                    (y + th < height and bottom > th - margin * height)):
                    continue
                if (right - left) * (bottom - top) <= spec["max_area"] * width * height:
                    rows.append([score, x + left, y + top, x + right, y + bottom])
    return fuse([rows], width, height, [1], spec["iou"])


def overlaps(boxes, other):
    lo = np.maximum(boxes[:, None, :2], other[None, :, :2])
    hi = np.minimum(boxes[:, None, 2:], other[None, :, 2:])
    joint = (hi - lo).clip(min=0).prod(axis=2)
    area = (boxes[:, 2:] - boxes[:, :2]).prod(axis=1)
    other_area = (other[:, 2:] - other[:, :2]).prod(axis=1)
    return joint / (area[:, None] + other_area[None, :] - joint)


def as_array(rows):
    return np.asarray(rows, dtype=np.float64).reshape(-1, 5)


def tile_rescue(baseline, tiles, spec):
    if not tiles:
        return baseline
    guesses = as_array(tiles)
    present = (overlaps(guesses[:, 1:], as_array(baseline)[:, 1:]).max(axis=1)
               if baseline else np.zeros(len(tiles)))
    return [*baseline, *guesses[(guesses[:, 0] >= spec["rescue_confidence"]) &
                               (present < spec["rescue_iou"])].tolist()]


def quality_features(candidates, sources, width, height):
    boxes = candidates[:, 1:]
    coords = boxes / [width, height, width, height]
    span = coords[:, 2:] - coords[:, :2]
    border = np.minimum(coords[:, :2], 1 - coords[:, 2:]).min(axis=1)
    columns = [candidates[:, 0], np.log(span[:, 0]), np.log(span[:, 1]), border]
    for rows in sources:
        raw = as_array(rows)
        support = np.zeros(len(candidates))
        agreement = np.zeros(len(candidates))
        if len(raw):
            similarity = overlaps(boxes, raw[:, 1:])
            best = similarity.argmax(axis=1)
            agreement = similarity[np.arange(len(best)), best]
            support = np.where(agreement >= 0.25, raw[best, 0], 0)
        columns.extend((agreement, support, agreement * support,
                        np.full(len(candidates), np.log1p(len(raw)))))
    return np.column_stack(columns)


def quality_candidates(global_rows, tiles, width, height, weights, threshold, with_tiles):
    sources = [*global_rows, tiles] if with_tiles else global_rows
    global_fused = fuse(global_rows, width, height, weights, threshold)
    # Keep alternative coordinates until the predicted quality determines NMS order.
    candidates = as_array([*global_fused, *(box for rows in sources for box in rows)])
    return candidates, quality_features(candidates, sources, width, height)


def rescore(candidates, features, width, height, spec):
    if not len(candidates):
        return []
    scores = np.clip(spec["intercept"] + features @ spec["coefficients"], 0, 1)
    remaining = np.argsort(-scores, kind="stable")
    remaining = remaining[scores[remaining] >= spec["confidence"]]
    selected = []
    while len(remaining):
        first, remaining = remaining[0], remaining[1:]
        selected.append(first)
        similarity = overlaps(candidates[[first], 1:], candidates[remaining, 1:])[0]
        remaining = remaining[similarity < spec["nms_iou"]]
    rows = np.column_stack((scores[selected], candidates[selected, 1:])).tolist()
    validate_predictions(rows, width, height)
    return rows


## submission.py

In [ ]:
%%writefile submission.py
"""Write Kaggle predictions from frozen detector checkpoints."""

import csv
import json
import math
from pathlib import Path

import torch
from PIL import Image
from torchvision.transforms.functional import pil_to_tensor

from data import sha
from fusion import fuse
from metric import validate_predictions
from model import create_model
from proposals import quality_candidates, rescore, tile_predictions, tile_rescue


def kaggle_inputs(mounted, spec):
    roots = sorted(path.parent for path in mounted.rglob("sample_submission.csv")
                   if path.parent.name == "global-wheat-detection")
    if len(roots) != 1:
        raise ValueError("Expected one mounted competition dataset.")
    by_hash = {sha(path): path for path in mounted.rglob("weights.pt")}
    return roots[0], [by_hash[model["sha256"]] for model in spec["models"]]


def prediction_string(predictions, width, height):
    validate_predictions(predictions, width, height)
    tokens = []
    for score, x1, y1, x2, y2 in predictions:
        left, top = math.floor(x1), math.floor(y1)
        right, bottom = math.ceil(x2), math.ceil(y2)
        tokens.append(f"{score:.8f} {left} {top} {right-left} {bottom-top}")
    return " ".join(tokens)


def predict_views(model, tensor, flips, iou):
    height, width = tensor.shape[-2:]
    predictions = []
    for view in ("original", *flips):
        if view == "original":
            result = model([tensor])[0]
            boxes = result["boxes"]
        else:
            dimension, extent = {"horizontal": (-1, width), "vertical": (-2, height)}[view]
            result = model([tensor.flip(dims=(dimension,))])[0]
            boxes = result["boxes"].clone()
            columns = (0, 2) if view == "horizontal" else (1, 3)
            boxes[:, columns] = extent - boxes[:, columns[::-1]]
        predictions.append([[score, *box] for score, box in
                            zip(result["scores"].tolist(), boxes.tolist(), strict=True)])
    return fuse(predictions, width, height, [1] * len(predictions), iou) if flips else predictions[0]


@torch.inference_mode()
def predict(data_root, checkpoints, spec, output, device):
    if len(checkpoints) != len(spec["models"]):
        raise ValueError("Expected one checkpoint per model.")
    models = []
    for path, conditions in zip(checkpoints, spec["models"], strict=True):
        if sha(path) != conditions["sha256"]:
            raise ValueError(f"Checkpoint mismatch for {conditions['trial_id']}.")
        model = create_model(conditions["config"], pretrained=False).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
        models.append(model.eval())
    paths = {path.stem: path for path in (data_root / "test").glob("*.jpg")}
    views = [conditions.get("tta_flips", spec.get("tta_flips", [])) for conditions in spec["models"]]
    for flips in views:
        if len(set(flips)) != len(flips) or set(flips) - {"horizontal", "vertical"}:
            raise ValueError("Expected a unique list of horizontal or vertical TTA flips.")
    with (data_root / "sample_submission.csv").open(newline="") as handle:
        ids = [row["image_id"] for row in csv.DictReader(handle)]
    if not ids or len(ids) != len(paths) or set(ids) != paths.keys():
        raise ValueError("Sample submission and test images must contain the same unique IDs.")
    temporary = output.with_suffix(".tmp")
    with temporary.open("w", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(("image_id", "PredictionString"))
        for key in ids:
            with Image.open(paths[key]) as image:
                width, height = image.size
                tensor = pil_to_tensor(image.convert("RGB")).float().div_(255).to(device)
            rows = [predict_views(model, tensor, flips, spec.get("tta_iou", 0.55))
                    for model, flips in zip(models, views, strict=True)]
            tiles = spec.get("tiles")
            tile_rows = tile_predictions(models[tiles["model_index"]], tensor, tiles) if tiles else []
            quality = spec.get("quality")
            if quality:
                candidates, features = quality_candidates(rows, tile_rows, width, height,
                                                         spec["weights"], spec["iou"], bool(tiles))
                detections = rescore(candidates, features, width, height, quality)
                writer.writerow((key, prediction_string(detections, width, height)))
                continue
            if len(rows) == 1:
                detections = rows[0]
            else:
                detections = fuse(rows, width, height, spec["weights"], spec["iou"])
            detections = [box for box in detections if box[0] >= spec["confidence"]]
            if tiles:
                detections = tile_rescue(detections, tile_rows, tiles)
            writer.writerow((key, prediction_string(detections, width, height)))
    temporary.replace(output)
    print(f"Validated and wrote {len(ids)} predictions")


def run_kaggle(spec_path):
    spec = json.loads(spec_path.read_text())
    data, checkpoints = kaggle_inputs(Path("/kaggle/input"), spec)
    torch.set_num_threads(2)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    predict(data, checkpoints, spec, Path("/kaggle/working/submission.csv"), device)


## 固定分割のタイル候補を保存する

In [ ]:
from PIL import Image
import torch
from torchvision.transforms.functional import pil_to_tensor

from data import sha
from model import create_model
from proposals import tile_predictions
from submission import kaggle_inputs

torch.set_num_threads(2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset, weights = kaggle_inputs(Path("/kaggle/input"), spec)
if sha(dataset / "train.csv") != spec["csv_sha256"]:
    raise ValueError("Competition annotations differ from the frozen split.")
conditions = spec["models"][spec["tiles"]["model_index"]]
checkpoint = weights[spec["tiles"]["model_index"]]
model = create_model(conditions["config"], pretrained=False).to(device).eval()
model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
rows = {}
with torch.inference_mode():
    for count, (key, expected) in enumerate(spec["images"].items(), 1):
        path = dataset / "train" / f"{key}.jpg"
        if sha(path) != expected:
            raise ValueError(f"Competition image does not match the frozen split: {key}")
        with Image.open(path) as image:
            tensor = pil_to_tensor(image.convert("RGB")).float().div_(255).to(device)
        rows[key] = tile_predictions(model, tensor, spec["tiles"])
        if count % 100 == 0:
            print(f"{count}/{len(spec['images'])} images")
meta = {"weights_sha256": conditions["sha256"], "config": conditions["config"],
        "tiles": spec["tiles"], "split_sha256": spec["split_sha256"],
        "torch": torch.__version__, "device": device.type, "supplement": None}
out = Path("/kaggle/working") / (spec["partition"] + "-tiles.json")
out.write_text(json.dumps({"meta": meta, "images": rows}, allow_nan=False) + "\n")
print(f"Wrote {len(rows)} proposal sets to {out.name}.")
